# Indian Market Trading — GPU Bootstrap

This notebook is the clean GPU runtime bootstrap. It materializes the required source files from the prepared GitHub branch into `/content/indian-market-trading`, while Google Drive remains the durable research store.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/Indian_Market_Research')
print('Drive mounted:', Path('/content/drive/MyDrive').exists())
print('Research root exists:', BASE.exists())
print('Research root:', BASE)


In [ ]:
from pathlib import Path

PROJECT = Path("/content/indian-market-trading")
PROJECT.mkdir(parents=True, exist_ok=True)

FILES = {"persistent_store.py":"\"\"\"Persistent historical market-data store.\n\nThe Streamlit container is treated as a cache, not as the system of record.\n1-minute observations are partitioned by IST trading date and mirrored to\nGoogle Drive using OAuth when available, with the existing service-account\nconfiguration retained as a fallback.\n\nConfiguration for Streamlit Cloud:\n\n[gdrive]\nroot_folder_id = \"<Google Drive root folder ID>\"\nservice_account_json = '''{...service-account-json...}'''\noauth_token_json = '''{...authorized-user-token-json...}'''\n\nGitHub Actions should provide GOOGLE_DRIVE_ROOT_ID and\nGOOGLE_OAUTH_TOKEN_JSON. The service-account variables remain supported as\na fallback for existing deployments.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport tempfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nimport pandas as pd\n\nBASE_DIR = Path(__file__).resolve().parent\nPERSIST_ROOT = BASE_DIR / \"data\" / \"persistent_market_data\"\nINTRADAY_ROOT = PERSIST_ROOT / \"intraday_1m\"\nMANIFEST_PATH = PERSIST_ROOT / \"manifest.json\"\nDRIVE_SCOPES = [\"https://www.googleapis.com/auth/drive\"]\n\n\ndef _now_utc() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef _safe_json_read(path: Path, default: Any) -> Any:\n    try:\n        return json.loads(path.read_text(encoding=\"utf-8\"))\n    except Exception:\n        return default\n\n\ndef _atomic_json(path: Path, obj: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, tmp = tempfile.mkstemp(prefix=path.name + \".\", suffix=\".tmp\", dir=str(path.parent))\n    try:\n        with os.fdopen(fd, \"w\", encoding=\"utf-8\") as fh:\n            json.dump(obj, fh, indent=2, sort_keys=True, default=str)\n            fh.flush()\n            os.fsync(fh.fileno())\n        os.replace(tmp, path)\n    finally:\n        if os.path.exists(tmp):\n            os.unlink(tmp)\n\n\ndef _sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open(\"rb\") as fh:\n        for block in iter(lambda: fh.read(1024 * 1024), b\"\"):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef ensure_dirs() -> None:\n    INTRADAY_ROOT.mkdir(parents=True, exist_ok=True)\n    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)\n\n\ndef load_manifest() -> dict:\n    ensure_dirs()\n    m = _safe_json_read(MANIFEST_PATH, {})\n    if not isinstance(m, dict):\n        m = {}\n    m.setdefault(\"version\", \"persistent-market-store-v1\")\n    m.setdefault(\"updated_at\", None)\n    m.setdefault(\"files\", {})\n    return m\n\n\ndef save_manifest(m: dict) -> None:\n    m[\"updated_at\"] = _now_utc()\n    _atomic_json(MANIFEST_PATH, m)\n\n\ndef normalize_symbol(symbol: str) -> str:\n    s = str(symbol).strip().upper()\n    if s.endswith(\".NS\") or s.endswith(\".BO\"):\n        s = s.rsplit(\".\", 1)[0]\n    return s.replace(\"/\", \"_\")\n\n\ndef _normalize_1m(df: pd.DataFrame) -> pd.DataFrame:\n    if df is None or df.empty:\n        return pd.DataFrame()\n    x = df.copy()\n    rename: dict[Any, str] = {}\n    for c in x.columns:\n        lc = str(c).strip().lower().replace(\" \", \"_\")\n        if lc in {\"timestamp\", \"datetime\", \"date\", \"time\", \"dt\", \"_time\"}:\n            rename[c] = \"_time\"\n        elif lc in {\"open\", \"o\"}:\n            rename[c] = \"Open\"\n        elif lc in {\"high\", \"h\"}:\n            rename[c] = \"High\"\n        elif lc in {\"low\", \"l\"}:\n            rename[c] = \"Low\"\n        elif lc in {\"close\", \"c\", \"ltp\"}:\n            rename[c] = \"Close\"\n        elif lc in {\"volume\", \"vol\", \"v\"}:\n            rename[c] = \"Volume\"\n    x = x.rename(columns=rename)\n    if \"_time\" in x.columns:\n        idx = pd.to_datetime(x.pop(\"_time\"), errors=\"coerce\")\n    else:\n        idx = pd.to_datetime(x.index, errors=\"coerce\")\n    if getattr(idx, \"tz\", None) is not None:\n        idx = idx.tz_convert(\"Asia/Kolkata\").tz_localize(None)\n    else:\n        idx = pd.DatetimeIndex(idx)\n    x.index = pd.DatetimeIndex(idx)\n    for c in (\"Open\", \"High\", \"Low\", \"Close\", \"Volume\"):\n        if c not in x.columns:\n            x[c] = np.nan\n        x[c] = pd.to_numeric(x[c], errors=\"coerce\")\n    x = x[~x.index.isna()].sort_index()\n    x = x[~x.index.duplicated(keep=\"last\")]\n    x = x.dropna(subset=[\"Open\", \"High\", \"Low\", \"Close\"])\n    return x\n\n\ndef _partition_path(symbol: str, day: pd.Timestamp) -> Path:\n    root = normalize_symbol(symbol)\n    d = pd.Timestamp(day).normalize()\n    return INTRADAY_ROOT / f\"{d:%Y}\" / f\"{d:%m}\" / f\"{d:%d}\" / f\"{root}.parquet\"\n\n\ndef local_partition_files(symbol: str | None = None) -> list[Path]:\n    ensure_dirs()\n    files = sorted(INTRADAY_ROOT.glob(\"*/*/*/*.parquet\"))\n    if symbol:\n        stem = normalize_symbol(symbol)\n        files = [p for p in files if p.stem.upper() == stem]\n    return files\n\n\ndef _relative_or_absolute(path: Path) -> str:\n    try:\n        return str(path.relative_to(BASE_DIR)).replace(\"\\\\\", \"/\")\n    except ValueError:\n        return str(path)\n\n\ndef gdrive_config() -> dict:\n    \"\"\"Resolve Google Drive configuration from environment or Streamlit Secrets.\n\n    Streamlit nested secrets may be represented by a mapping/AttrDict rather\n    than a plain Python dict, so configuration detection must not require\n    isinstance(section, dict).\n    \"\"\"\n    root_id = (\n        os.getenv(\"GDRIVE_ROOT_FOLDER_ID\", \"\").strip()\n        or os.getenv(\"GOOGLE_DRIVE_ROOT_ID\", \"\").strip()\n        or os.getenv(\"GOOGLE_DRIVE_ROOT_FOLDER_ID\", \"\").strip()\n        or os.getenv(\"DRIVE_ROOT_FOLDER_ID\", \"\").strip()\n    )\n\n    root_name = (\n        os.getenv(\"GOOGLE_DRIVE_ROOT_FOLDER_NAME\", \"\").strip()\n        or os.getenv(\"DRIVE_ROOT_FOLDER_NAME\", \"\").strip()\n        or \"Indian_Market_Research\"\n    )\n\n    service_json = os.getenv(\n        \"GOOGLE_SERVICE_ACCOUNT_JSON\",\n        \"\",\n    ).strip()\n\n    oauth_json = os.getenv(\n        \"GOOGLE_OAUTH_TOKEN_JSON\",\n        \"\",\n    ).strip()\n\n    secrets_seen = False\n    secret_names = []\n\n    try:\n        import streamlit as st\n\n        if hasattr(st, \"secrets\"):\n\n            # Record only key names. Never expose secret values.\n            try:\n                secret_names = [\n                    str(k)\n                    for k in st.secrets.keys()\n                ]\n                secrets_seen = True\n            except Exception:\n                pass\n\n            # Read structured sections without assuming plain dict.\n            for section_name in (\n                \"gdrive\",\n                \"google_drive\",\n                \"drive\",\n            ):\n                try:\n                    if section_name not in st.secrets:\n                        continue\n\n                    sec = st.secrets[section_name]\n\n                    def sec_value(key):\n                        try:\n                            return str(\n                                sec[key]\n                            ).strip()\n                        except Exception:\n                            return \"\"\n\n                    value = sec_value(\n                        \"root_folder_id\"\n                    )\n                    if value and not root_id:\n                        root_id = value\n\n                    value = sec_value(\n                        \"root_folder_name\"\n                    )\n                    if value:\n                        root_name = value\n\n                    value = sec_value(\n                        \"service_account_json\"\n                    )\n                    if value and not service_json:\n                        service_json = value\n\n                    value = sec_value(\n                        \"oauth_token_json\"\n                    )\n                    if value and not oauth_json:\n                        oauth_json = value\n\n                except Exception:\n                    continue\n\n            # Also support top-level secrets used by previous deployments.\n            def top_secret(key):\n                try:\n                    return str(\n                        st.secrets[key]\n                    ).strip()\n                except Exception:\n                    return \"\"\n\n            for key in (\n                \"GOOGLE_DRIVE_ROOT_ID\",\n                \"GDRIVE_ROOT_FOLDER_ID\",\n                \"GOOGLE_DRIVE_ROOT_FOLDER_ID\",\n                \"DRIVE_ROOT_FOLDER_ID\",\n                \"ROOT_FOLDER_ID\",\n            ):\n                if not root_id:\n                    value = top_secret(key)\n                    if value:\n                        root_id = value\n\n            for key in (\n                \"GOOGLE_DRIVE_ROOT_FOLDER_NAME\",\n                \"DRIVE_ROOT_FOLDER_NAME\",\n                \"ROOT_FOLDER_NAME\",\n            ):\n                value = top_secret(key)\n                if value:\n                    root_name = value\n                    break\n\n            for key in (\n                \"GOOGLE_SERVICE_ACCOUNT_JSON\",\n                \"SERVICE_ACCOUNT_JSON\",\n            ):\n                if not service_json:\n                    value = top_secret(key)\n                    if value:\n                        service_json = value\n\n            for key in (\n                \"GOOGLE_OAUTH_TOKEN_JSON\",\n                \"OAUTH_TOKEN_JSON\",\n            ):\n                if not oauth_json:\n                    value = top_secret(key)\n                    if value:\n                        oauth_json = value\n\n    except Exception:\n        pass\n\n    if oauth_json:\n        try:\n            oauth_info = json.loads(oauth_json)\n        except Exception as exc:\n            return {\n                \"configured\": False,\n                \"reason\": (\n                    f\"invalid OAuth token JSON: \"\n                    f\"{type(exc).__name__}\"\n                ),\n                \"secrets_detected\": secrets_seen,\n                \"secret_keys\": secret_names,\n            }\n\n        if not isinstance(oauth_info, dict):\n            return {\n                \"configured\": False,\n                \"reason\": \"OAuth token JSON is not an object\",\n                \"secrets_detected\": secrets_seen,\n                \"secret_keys\": secret_names,\n            }\n\n        if not oauth_info.get(\"refresh_token\"):\n            return {\n                \"configured\": False,\n                \"reason\": \"OAuth token JSON has no refresh_token\",\n                \"secrets_detected\": secrets_seen,\n                \"secret_keys\": secret_names,\n            }\n\n        return {\n            \"configured\": True,\n            \"auth_method\": \"oauth\",\n            \"root_folder_id\": root_id,\n            \"root_folder_name\": root_name,\n            \"oauth_token_info\": oauth_info,\n            \"secrets_detected\": secrets_seen,\n            \"secret_keys\": secret_names,\n        }\n\n    if not service_json:\n        return {\n            \"configured\": False,\n            \"reason\": (\n                \"missing OAuth token JSON and \"\n                \"service-account JSON\"\n            ),\n            \"secrets_detected\": secrets_seen,\n            \"secret_keys\": secret_names,\n            \"root_folder_id_present\": bool(root_id),\n            \"root_folder_name\": root_name,\n        }\n\n    try:\n        info = json.loads(service_json)\n    except Exception as exc:\n        return {\n            \"configured\": False,\n            \"reason\": (\n                f\"invalid service-account JSON: \"\n                f\"{type(exc).__name__}\"\n            ),\n            \"secrets_detected\": secrets_seen,\n            \"secret_keys\": secret_names,\n        }\n\n    if not isinstance(info, dict):\n        return {\n            \"configured\": False,\n            \"reason\": \"service-account JSON is not an object\",\n            \"secrets_detected\": secrets_seen,\n            \"secret_keys\": secret_names,\n        }\n\n    if not info.get(\"client_email\"):\n        return {\n            \"configured\": False,\n            \"reason\": \"service-account JSON has no client_email\",\n            \"secrets_detected\": secrets_seen,\n            \"secret_keys\": secret_names,\n        }\n\n    return {\n        \"configured\": True,\n        \"auth_method\": \"service_account\",\n        \"root_folder_id\": root_id,\n        \"root_folder_name\": root_name,\n        \"service_account_info\": info,\n        \"client_email\": info.get(\"client_email\"),\n        \"secrets_detected\": secrets_seen,\n        \"secret_keys\": secret_names,\n    }\n\n\ndef _drive_credentials(cfg: dict):\n    \"\"\"Build authenticated Drive credentials using OAuth first, then service account.\"\"\"\n    if cfg.get(\"auth_method\") == \"oauth\":\n        from google.auth.transport.requests import Request\n        from google.oauth2.credentials import Credentials\n\n        credentials = Credentials.from_authorized_user_info(\n            cfg[\"oauth_token_info\"],\n            scopes=DRIVE_SCOPES,\n        )\n\n        if not credentials.valid:\n            if not credentials.refresh_token:\n                raise RuntimeError(\n                    \"OAuth credentials are invalid and have no refresh token\"\n                )\n            credentials.refresh(Request())\n\n        return credentials\n\n    from google.oauth2 import service_account\n\n    return service_account.Credentials.from_service_account_info(\n        cfg[\"service_account_info\"],\n        scopes=DRIVE_SCOPES,\n    )\n\n\ndef drive_status() -> dict:\n    cfg = gdrive_config()\n    if not cfg.get(\"configured\"):\n        return {\"configured\": False, \"reason\": cfg.get(\"reason\"), \"persistent_local_root\": _relative_or_absolute(PERSIST_ROOT)}\n    try:\n        from googleapiclient.discovery import build\n\n        credentials = _drive_credentials(cfg)\n        service = build(\n            \"drive\",\n            \"v3\",\n            credentials=credentials,\n            cache_discovery=False,\n        )\n\n        root_id = str(\n            cfg.get(\"root_folder_id\", \"\")\n        ).strip()\n\n        if not root_id:\n            root_id = _discover_drive_root_folder(\n                service,\n                cfg.get(\n                    \"root_folder_name\",\n                    \"Indian_Market_Research\",\n                ),\n            )\n\n        meta = service.files().get(\n            fileId=root_id,\n            fields=\"id,name,mimeType,trashed\",\n        ).execute()\n        if meta.get(\"trashed\"):\n            return {\"configured\": False, \"reason\": \"configured root folder is in Trash\"}\n        if meta.get(\"mimeType\") != \"application/vnd.google-apps.folder\":\n            return {\"configured\": False, \"reason\": \"configured root ID is not a folder\"}\n        return {\n            \"configured\": True,\n            \"root_folder_id\": root_id,\n            \"auth_method\": cfg.get(\"auth_method\"),\n            \"client_email\": cfg.get(\"client_email\"),\n            \"root_name\": meta.get(\"name\"),\n        }\n    except Exception as exc:\n        return {\"configured\": True, \"reachable\": False, \"reason\": f\"Google Drive check failed: {type(exc).__name__}: {exc}\", \"client_email\": cfg.get(\"client_email\")}\n\n\ndef _discover_drive_root_folder(\n    service,\n    folder_name: str,\n) -> str:\n    \"\"\"Discover an exact folder name in the connected Google Drive.\"\"\"\n    name = str(\n        folder_name or \"Indian_Market_Research\"\n    ).strip()\n\n    if not name:\n        raise RuntimeError(\n            \"Google Drive root folder name is empty.\"\n        )\n\n    q = (\n        f\"name = '{name.replace(chr(39), chr(39) + chr(39))}' \"\n        \"and mimeType = \"\n        \"'application/vnd.google-apps.folder' \"\n        \"and trashed = false\"\n    )\n\n    response = service.files().list(\n        q=q,\n        spaces=\"drive\",\n        fields=\"files(id,name,mimeType,parents,driveId)\",\n        pageSize=20,\n        orderBy=\"name\",\n        includeItemsFromAllDrives=True,\n        supportsAllDrives=True,\n    ).execute()\n\n    matches = response.get(\n        \"files\",\n        [],\n    )\n\n    if not matches:\n        raise RuntimeError(\n            \"Google Drive research folder was not found: \"\n            f\"{name!r}. Either share the folder with the configured \"\n            \"Google identity or provide its root_folder_id explicitly.\"\n        )\n\n    # Exact-name match; deterministic first item.\n    return str(\n        sorted(\n            matches,\n            key=lambda x: (\n                str(x.get(\"name\", \"\")),\n                str(x.get(\"id\", \"\")),\n            ),\n        )[0][\"id\"]\n    )\n\n\nclass GoogleDriveStore:\n    def __init__(self) -> None:\n        cfg = gdrive_config()\n        if not cfg.get(\"configured\"):\n            raise RuntimeError(cfg.get(\"reason\", \"Google Drive is not configured\"))\n        try:\n            from googleapiclient.discovery import build\n            from googleapiclient.http import MediaFileUpload\n        except Exception as exc:\n            raise RuntimeError(\"Google Drive dependencies are not installed\") from exc\n        self.credentials = _drive_credentials(cfg)\n        self.service = build(\n            \"drive\",\n            \"v3\",\n            credentials=self.credentials,\n            cache_discovery=False,\n        )\n\n        configured_root = str(\n            cfg.get(\"root_folder_id\", \"\")\n        ).strip()\n\n        if configured_root:\n            self.root_id = configured_root\n        else:\n            self.root_id = _discover_drive_root_folder(\n                self.service,\n                cfg.get(\n                    \"root_folder_name\",\n                    \"Indian_Market_Research\",\n                ),\n            )\n\n        self.MediaFileUpload = MediaFileUpload\n\n    @staticmethod\n    def _q(value: str) -> str:\n        return value.replace(\"\\\\\", \"\\\\\\\\\").replace(\"'\", \"\\\\'\")\n\n    def _find_child(self, parent_id: str, name: str, mime_type: str | None = None) -> dict | None:\n        q = f\"name = '{self._q(name)}' and '{parent_id}' in parents and trashed = false\"\n        if mime_type:\n            q += f\" and mimeType = '{mime_type}'\"\n        resp = self.service.files().list(\n            q=q,\n            spaces=\"drive\",\n            fields=\"files(id,name,mimeType)\",\n            pageSize=10,\n            includeItemsFromAllDrives=True,\n            supportsAllDrives=True,\n        ).execute()\n        files = resp.get(\"files\", [])\n        return files[0] if files else None\n\n    def _ensure_folder(self, parent_id: str, name: str) -> str:\n        hit = self._find_child(parent_id, name, \"application/vnd.google-apps.folder\")\n        if hit:\n            return hit[\"id\"]\n        meta = {\"name\": name, \"mimeType\": \"application/vnd.google-apps.folder\", \"parents\": [parent_id]}\n        return self.service.files().create(body=meta, fields=\"id\").execute()[\"id\"]\n\n    def ensure_path(self, relative_folder: str) -> str:\n        parent = self.root_id\n        parts = [p for p in relative_folder.replace(\"\\\\\", \"/\").split(\"/\") if p]\n        for part in parts:\n            parent = self._ensure_folder(parent, part)\n        return parent\n\n    def upload_or_update(self, local_path: Path, remote_relative_path: str, mime_type: str = \"application/octet-stream\") -> dict:\n        remote = remote_relative_path.replace(\"\\\\\", \"/\").strip(\"/\")\n        parts = remote.split(\"/\")\n        name = parts[-1]\n        folder = \"/\".join(parts[:-1])\n        parent_id = self.ensure_path(folder)\n        existing = self._find_child(parent_id, name)\n        media = self.MediaFileUpload(str(local_path), mimetype=mime_type, resumable=True)\n        if existing:\n            result = self.service.files().update(fileId=existing[\"id\"], media_body=media, fields=\"id,name,modifiedTime,size\").execute()\n            action = \"updated\"\n        else:\n            result = self.service.files().create(body={\"name\": name, \"parents\": [parent_id]}, media_body=media, fields=\"id,name,createdTime,size\").execute()\n            action = \"created\"\n        return {\"status\": \"OK\", \"action\": action, \"remote_path\": remote, \"file_id\": result.get(\"id\"), \"size\": result.get(\"size\")}\n\n\ndef persist_1m_frame(symbol: str, df: pd.DataFrame, source: str = \"unknown\", sync_drive: bool = True) -> dict:\n    \"\"\"Merge 1-minute observations into date partitions and optionally mirror to Drive.\"\"\"\n    ensure_dirs()\n    x = _normalize_1m(df)\n    if x.empty:\n        return {\"status\": \"NO_DATA\", \"symbol\": normalize_symbol(symbol), \"rows_added\": 0, \"rows_total\": 0, \"partitions\": []}\n    root = normalize_symbol(symbol)\n    manifest = load_manifest()\n    partitions = []\n    total_added = 0\n    total_rows = 0\n    for day, part in x.groupby(x.index.normalize()):\n        path = _partition_path(root, day)\n        before_idx: set[int] | None = None\n        if path.exists():\n            try:\n                old = pd.read_parquet(path)\n                old = _normalize_1m(old)\n                before_idx = set(pd.Index(old.index).view(\"int64\"))\n                merged = pd.concat([old, part]).sort_index()\n                merged = merged[~merged.index.duplicated(keep=\"last\")]\n            except Exception:\n                merged = part.sort_index()\n        else:\n            merged = part.sort_index()\n        path.parent.mkdir(parents=True, exist_ok=True)\n        merged.to_parquet(path, index=True)\n        idx_after = set(pd.Index(merged.index).view(\"int64\"))\n        added = len(idx_after - (before_idx or set()))\n        total_added += added\n        total_rows += len(merged)\n        rel = str(path.relative_to(BASE_DIR)).replace(\"\\\\\", \"/\")\n        key = f\"intraday_1m:{root}:{day:%Y-%m-%d}\"\n        entry = {\n            \"kind\": \"intraday_1m_daily_partition\",\n            \"symbol\": root,\n            \"date\": f\"{day:%Y-%m-%d}\",\n            \"source\": source,\n            \"path\": rel,\n            \"rows\": int(len(merged)),\n            \"rows_added_last_write\": int(added),\n            \"sha256\": _sha256(path),\n            \"updated_at\": _now_utc(),\n            \"drive_sync\": \"pending\",\n        }\n        if sync_drive:\n            try:\n                g = GoogleDriveStore()\n                remote = f\"production_data/intraday_1m/{day:%Y}/{day:%m}/{day:%d}/{root}.parquet\"\n                out = g.upload_or_update(path, remote)\n                entry[\"drive_sync\"] = \"OK\"\n                entry[\"drive\"] = out\n            except Exception as exc:\n                entry[\"drive_sync\"] = \"NOT_SYNCED\"\n                entry[\"drive_error\"] = f\"{type(exc).__name__}: {exc}\"\n        manifest[\"files\"][key] = entry\n        partitions.append(entry)\n    save_manifest(manifest)\n    return {\"status\": \"OK\", \"symbol\": root, \"rows_added\": total_added, \"rows_total\": total_rows, \"partitions\": partitions}\n\n\ndef persist_local_1m_file(path: str | Path, symbol: str, source: str = \"local-file\", sync_drive: bool = True) -> dict:\n    p = Path(path)\n    if not p.exists():\n        raise FileNotFoundError(p)\n    df = pd.read_parquet(p)\n    return persist_1m_frame(symbol, df, source=source, sync_drive=sync_drive)\n\n\ndef load_persistent_1m(symbol: str, start=None, end=None) -> pd.DataFrame:\n    \"\"\"Load merged persistent daily partitions for a symbol.\"\"\"\n    files = local_partition_files(symbol)\n    if not files:\n        return pd.DataFrame()\n    start_ts = pd.Timestamp(start) if start is not None else None\n    end_ts = pd.Timestamp(end) if end is not None else None\n    frames = []\n    for path in files:\n        try:\n            day = pd.Timestamp(path.parent.name + \"-\" + path.parent.parent.name + \"-\" + path.parent.parent.parent.name)\n        except Exception:\n            day = None\n        if day is not None:\n            if start_ts is not None and end_ts is not None and (day.normalize() < start_ts.normalize() or day.normalize() > end_ts.normalize()):\n                continue\n            if start_ts is not None and end_ts is None and day.normalize() < start_ts.normalize():\n                continue\n            if end_ts is not None and start_ts is None and day.normalize() > end_ts.normalize():\n                continue\n        try:\n            x = pd.read_parquet(path)\n            x = _normalize_1m(x)\n            if start_ts is not None:\n                x = x[x.index >= start_ts]\n            if end_ts is not None:\n                x = x[x.index <= end_ts]\n            if not x.empty:\n                frames.append(x)\n        except Exception:\n            continue\n    if not frames:\n        return pd.DataFrame()\n    out = pd.concat(frames).sort_index()\n    return out[~out.index.duplicated(keep=\"last\")]\n\n\ndef scan_coverage(symbols: Iterable[str] | None = None) -> pd.DataFrame:\n    files = local_partition_files()\n    wanted = {normalize_symbol(s) for s in symbols} if symbols else None\n    rows = []\n    for path in files:\n        root = path.stem.upper()\n        if wanted and root not in wanted:\n            continue\n        try:\n            x = pd.read_parquet(path)\n            x = _normalize_1m(x)\n            rows.append({\n                \"dataset\": \"persistent_intraday_1m\",\n                \"symbol\": root,\n                \"trading_date\": path.parent.name,\n                \"month\": path.parent.parent.name,\n                \"year\": path.parent.parent.parent.name,\n                \"rows\": int(len(x)),\n                \"first_timestamp\": str(x.index.min()) if not x.empty else None,\n                \"last_timestamp\": str(x.index.max()) if not x.empty else None,\n                \"file_mb\": round(path.stat().st_size / (1024 * 1024), 3),\n                \"path\": str(path.relative_to(BASE_DIR)).replace(\"\\\\\", \"/\"),\n            })\n        except Exception as exc:\n            rows.append({\"dataset\": \"persistent_intraday_1m\", \"symbol\": root, \"path\": str(path), \"error\": f\"{type(exc).__name__}: {exc}\"})\n    return pd.DataFrame(rows)\n\n\ndef sync_pending(max_files: int = 100) -> dict:\n    \"\"\"Mirror local persistent partitions to Drive. Existing remote files are updated.\"\"\"\n    files = local_partition_files()[:max_files]\n    if not files:\n        return {\"status\": \"NO_DATA\", \"attempted\": 0, \"synced\": 0, \"errors\": []}\n    try:\n        g = GoogleDriveStore()\n    except Exception as exc:\n        return {\"status\": \"NOT_CONFIGURED\", \"attempted\": len(files), \"synced\": 0, \"errors\": [f\"{type(exc).__name__}: {exc}\"]}\n    synced = 0\n    errors = []\n    manifest = load_manifest()\n    for path in files:\n        rel = path.relative_to(INTRADAY_ROOT)\n        if len(rel.parts) != 4:\n            continue\n        year, month, day, name = rel.parts\n        symbol = Path(name).stem\n        remote = f\"production_data/intraday_1m/{year}/{month}/{day}/{symbol}.parquet\"\n        key = f\"intraday_1m:{symbol}:{year}-{month}-{day}\"\n        try:\n            out = g.upload_or_update(path, remote)\n            synced += 1\n            if key in manifest.get(\"files\", {}):\n                manifest[\"files\"][key][\"drive_sync\"] = \"OK\"\n                manifest[\"files\"][key][\"drive\"] = out\n        except Exception as exc:\n            errors.append({\"path\": str(path), \"error\": f\"{type(exc).__name__}: {exc}\"})\n    save_manifest(manifest)\n    return {\"status\": \"OK\" if not errors else \"PARTIAL\", \"attempted\": len(files), \"synced\": synced, \"errors\": errors}\n\n\ndef storage_summary() -> dict:\n    ensure_dirs()\n    total = 0\n    files = 0\n    for p in INTRADAY_ROOT.rglob(\"*.parquet\"):\n        files += 1\n        total += p.stat().st_size\n    m = load_manifest()\n    drive = drive_status()\n    return {\n        \"local_root\": _relative_or_absolute(PERSIST_ROOT),\n        \"local_files\": files,\n        \"local_gb\": round(total / (1024 ** 3), 4),\n        \"manifest_files\": len(m.get(\"files\", {})),\n        \"google_drive\": drive,\n    }\n","nse_data_layer.py":"\"\"\"NSE-first data layer for the Indian market dashboard.\n\nPrimary source: NSE India's public web data interfaces.\n\nThis module deliberately separates four concerns:\n1. NSE acquisition (official exchange web interfaces).\n2. Normalization into stable internal schemas.\n3. Local cache / persistence.\n4. Provenance metadata.\n\nIt does not bypass authentication, scrape CAPTCHAs, or attempt to defeat rate\nlimits. Requests are serialized by default, cached, and retried conservatively.\nFor production/licensed redistribution of NSE market data, follow NSE's current\nData Sharing & Usage Policy.\n\"\"\"\nfrom __future__ import annotations\n\nimport io\nimport json\nimport time\nimport zipfile\nfrom datetime import date, datetime, timedelta\nfrom pathlib import Path\nfrom typing import Any, Iterable\nfrom zoneinfo import ZoneInfo\n\nimport numpy as np\nimport pandas as pd\nimport requests\n\nNSE_BASE = \"https://www.nseindia.com\"\nIST = ZoneInfo(\"Asia/Kolkata\")\n\nHEADERS = {\n    \"User-Agent\": (\n        \"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 \"\n        \"(KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36\"\n    ),\n    \"Accept\": \"text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8\",\n    \"Accept-Language\": \"en-IN,en-US;q=0.9,en;q=0.8\",\n    \"Accept-Encoding\": \"gzip, deflate, br\",\n    \"Cache-Control\": \"no-cache\",\n    \"Pragma\": \"no-cache\",\n    \"Referer\": \"https://www.nseindia.com/report-detail/eq_security\",\n    \"Connection\": \"keep-alive\",\n    \"Sec-Fetch-Dest\": \"document\",\n    \"Sec-Fetch-Mode\": \"navigate\",\n    \"Sec-Fetch-Site\": \"none\",\n    \"Sec-Fetch-User\": \"?1\",\n    \"Upgrade-Insecure-Requests\": \"1\",\n}\n\n\nEQUITY_HISTORY_ENDPOINT = \"/api/historicalOR/generateSecurityWiseHistoricalData\"\nEQUITY_QUOTE_ENDPOINT = \"/api/quote-equity\"\nEQUITY_CHART_ENDPOINT = \"/api/chart-databyindex\"\nCORPORATE_ACTIONS_ENDPOINT = \"/api/corporates-corporateActions\"\nCORPORATE_ANNOUNCEMENTS_ENDPOINT = \"/api/corporate-announcements\"\nFNO_HISTORY_ENDPOINT = \"/api/historicalOR/foCPV\"\n\nLOCAL_ROOT = Path(__file__).resolve().parent / \"data\" / \"nse_official\"\nRAW_ROOT = LOCAL_ROOT / \"raw\"\nNORMALIZED_ROOT = LOCAL_ROOT / \"normalized\"\nMANIFEST = LOCAL_ROOT / \"manifest.json\"\n\n_last_request = 0.0\n_nse_session: requests.Session | None = None\n_nse_session_at = 0.0\n_last_nse_diagnostic: dict[str, Any] = {}\n\n\ndef root_symbol(symbol: str) -> str:\n    s = str(symbol).strip().upper()\n    for suffix in (\".NS\", \".BO\"):\n        if s.endswith(suffix):\n            s = s[: -len(suffix)]\n    return s.replace(\"/\", \"_\")\n\n\ndef _sleep_rate_limit(min_interval: float = 1.5) -> None:\n    global _last_request\n    now = time.monotonic()\n    wait = min_interval - (now - _last_request)\n    if wait > 0:\n        time.sleep(wait)\n    _last_request = time.monotonic()\n\n\ndef _session() -> requests.Session:\n    global _nse_session, _nse_session_at\n    now = time.monotonic()\n    if _nse_session is not None and (now - _nse_session_at) < 600 and _nse_session.cookies:\n        return _nse_session\n    s = requests.Session()\n    s.headers.update(HEADERS)\n    # Prime both the site root and the historical-report page so the cookies\n    # and anti-bot context expected by current NSE endpoints are established.\n    for url in (NSE_BASE, NSE_BASE + \"/report-detail/eq_security\"):\n        try:\n            s.get(url, timeout=4)\n        except requests.RequestException:\n            pass\n    _nse_session = s\n    _nse_session_at = time.monotonic()\n    return s\n\n\ndef _request(path: str, *, params: dict[str, Any] | None = None,\n             accept: str | None = None, retries: int = 3,\n             min_interval: float = 1.5) -> requests.Response:\n    global _last_nse_diagnostic, _nse_session, _nse_session_at\n    last_exc: Exception | None = None\n    for attempt in range(retries):\n        try:\n            _sleep_rate_limit(min_interval)\n            s = _session()\n            headers = {\n                \"Accept\": accept or \"application/json,text/plain,*/*\",\n                \"Referer\": \"https://www.nseindia.com/report-detail/eq_security\",\n                \"X-Requested-With\": \"XMLHttpRequest\",\n                \"Sec-Fetch-Dest\": \"empty\",\n                \"Sec-Fetch-Mode\": \"cors\",\n                \"Sec-Fetch-Site\": \"same-origin\",\n            }\n            r = s.get(NSE_BASE + path, params=params, headers=headers, timeout=(3, 8))\n            _last_nse_diagnostic = {\n                \"path\": path, \"status_code\": r.status_code,\n                \"content_type\": r.headers.get(\"content-type\", \"\"),\n                \"bytes\": len(r.content), \"attempt\": attempt + 1,\n            }\n            if r.status_code == 200:\n                return r\n            if r.status_code in (401, 403, 408, 429, 500, 502, 503, 504):\n                # Refresh the cookie session before the next attempt.\n                _nse_session = None\n                _nse_session_at = 0.0\n                time.sleep(min(8.0, 1.5 * (attempt + 1)))\n                continue\n            r.raise_for_status()\n        except Exception as exc:\n            last_exc = exc\n            _last_nse_diagnostic = {\n                \"path\": path, \"attempt\": attempt + 1,\n                \"error\": f\"{type(exc).__name__}: {exc}\",\n            }\n            _nse_session = None\n            _nse_session_at = 0.0\n            if attempt + 1 < retries:\n                time.sleep(min(8.0, 1.5 * (attempt + 1)))\n    if last_exc:\n        raise RuntimeError(f\"NSE request failed: {type(last_exc).__name__}: {last_exc}\") from last_exc\n    raise RuntimeError(f\"NSE request failed after {retries} attempts: {path}\")\n\n\ndef nse_last_diagnostic() -> dict[str, Any]:\n    \"\"\"Return the last NSE transport diagnostic without exposing response bodies.\"\"\"\n    return dict(_last_nse_diagnostic)\n\n\ndef _ensure_local() -> None:\n    for p in (RAW_ROOT, NORMALIZED_ROOT):\n        p.mkdir(parents=True, exist_ok=True)\n\n\ndef _manifest() -> dict[str, Any]:\n    _ensure_local()\n    try:\n        obj = json.loads(MANIFEST.read_text(encoding=\"utf-8\"))\n        if isinstance(obj, dict):\n            obj.setdefault(\"records\", {})\n            return obj\n    except Exception:\n        pass\n    return {\"version\": \"nse-official-v1\", \"records\": {}}\n\n\ndef _save_manifest(obj: dict[str, Any]) -> None:\n    obj[\"updated_at\"] = datetime.now(tz=IST).isoformat()\n    _ensure_local()\n    tmp = MANIFEST.with_suffix(\".json.tmp\")\n    tmp.write_text(\n        json.dumps(obj, indent=2, sort_keys=True, default=str),\n        encoding=\"utf-8\",\n    )\n    tmp.replace(MANIFEST)\n\n\ndef _drive_upload(local_path: Path, remote_path: str, mime: str = \"application/octet-stream\") -> dict[str, Any]:\n    try:\n        from persistent_store import GoogleDriveStore\n        return GoogleDriveStore().upload_or_update(local_path, remote_path, mime_type=mime)\n    except Exception as exc:\n        return {\"status\": \"NOT_SYNCED\", \"error\": f\"{type(exc).__name__}: {exc}\"}\n\n\ndef normalize_equity_history(frame: pd.DataFrame, symbol: str) -> pd.DataFrame:\n    \"\"\"Normalize current NSE historical equity CSV/JSON into stable columns.\"\"\"\n    if frame is None or frame.empty:\n        return pd.DataFrame()\n    x = frame.copy()\n    x.columns = [str(c).strip() for c in x.columns]\n    aliases = {\n        \"Date\": \"date\", \"DATE\": \"date\", \"CH_TIMESTAMP\": \"date\",\n        \"OPEN\": \"open\", \"CH_OPENING_PRICE\": \"open\",\n        \"HIGH\": \"high\", \"CH_TRADE_HIGH_PRICE\": \"high\",\n        \"LOW\": \"low\", \"CH_TRADE_LOW_PRICE\": \"low\",\n        \"CLOSE\": \"close\", \"CH_CLOSING_PRICE\": \"close\",\n        \"LTP\": \"ltp\", \"CH_LAST_TRADED_PRICE\": \"ltp\",\n        \"PREV. CLOSE\": \"prev_close\", \"PREV CLOSE\": \"prev_close\", \"CH_PREVIOUS_CLS_PRICE\": \"prev_close\",\n        \"VWAP\": \"vwap\", \"VOLUME\": \"volume\", \"CH_TOT_TRADED_QTY\": \"volume\",\n        \"VALUE\": \"traded_value\", \"CH_TOT_TRADED_VAL\": \"traded_value\",\n        \"NO OF TRADES\": \"num_trades\", \"CH_TOTAL_TRADES\": \"num_trades\",\n        \"DELIVERY QTY\": \"delivery_qty\", \"COP_DELIV_QTY\": \"delivery_qty\",\n        \"DELIVERY %\": \"delivery_pct\", \"COP_DELIV_PERC\": \"delivery_pct\",\n        \"CH_SERIES\": \"series\", \"CH_SYMBOL\": \"symbol\",\n    }\n    ren = {c: aliases[c] for c in x.columns if c in aliases}\n    x = x.rename(columns=ren)\n    if \"symbol\" not in x.columns:\n        x[\"symbol\"] = root_symbol(symbol)\n    x[\"symbol\"] = x[\"symbol\"].astype(str).str.upper()\n    if \"date\" not in x.columns:\n        raise ValueError(f\"NSE equity response has no date column: {list(x.columns)}\")\n    x[\"date\"] = pd.to_datetime(x[\"date\"], errors=\"coerce\", format=\"mixed\", dayfirst=True)\n    for c in (\"open\", \"high\", \"low\", \"close\", \"ltp\", \"prev_close\", \"vwap\", \"volume\", \"traded_value\", \"num_trades\", \"delivery_qty\", \"delivery_pct\"):\n        if c in x.columns:\n            x[c] = pd.to_numeric(x[c].astype(str).str.replace(\",\", \"\", regex=False), errors=\"coerce\")\n    if \"close\" not in x.columns and \"ltp\" in x.columns:\n        x[\"close\"] = x[\"ltp\"]\n    x[\"source\"] = \"NSE_OFFICIAL\"\n    x[\"provenance\"] = \"nseindia.com public historical equity interface\"\n    return x.sort_values(\"date\").drop_duplicates([\"symbol\", \"date\"], keep=\"last\").reset_index(drop=True)\n\n\ndef fetch_equity_history(symbol: str, start: date | datetime, end: date | datetime,\n                         series: str = \"EQ\", sync_drive: bool = True) -> pd.DataFrame:\n    \"\"\"Fetch official NSE security-wise historical price/volume data.\n\n    Uses the same public report data exposed by NSE's Price/Volume security\n    interface, returned as CSV. The request is chunked to reduce timeout/rate-\n    limiting risk and persisted locally; each chunk is also mirrored to Drive\n    when Google Drive is configured.\n    \"\"\"\n    s = root_symbol(symbol)\n    start_d = pd.Timestamp(start).date()\n    end_d = pd.Timestamp(end).date()\n    frames: list[pd.DataFrame] = []\n    cursor = start_d\n    while cursor <= end_d:\n        chunk_end = min(cursor + timedelta(days=89), end_d)\n        base_params = {\n            \"from\": cursor.strftime(\"%d-%m-%Y\"),\n            \"to\": chunk_end.strftime(\"%d-%m-%Y\"),\n            \"symbol\": s,\n            \"type\": \"priceVolumeDeliverable\",\n            # The security-wise historical NSE interface uses csv=true for\n            # the actual tabular download response. Without it, the endpoint\n            # can return HTTP 200 with a JSON response that is not a usable\n            # historical row set on hosted runtimes.\n            \"csv\": \"true\",\n        }\n\n        # Prefer the explicit CSV contract, then retry without the csv hint\n        # as a compatibility fallback, and finally try ALL-series.\n        request_variants = [\n            {**base_params, \"series\": series},\n            {**base_params, \"series\": \"ALL\"},\n            {k: v for k, v in base_params.items() if k != \"csv\"} | {\"series\": series},\n            {k: v for k, v in base_params.items() if k != \"csv\"} | {\"series\": \"ALL\"},\n        ]\n        r = None\n        csv = pd.DataFrame()\n        for params in request_variants:\n            try:\n                rr = _request(EQUITY_HISTORY_ENDPOINT, params=params,\n                               accept=\"application/json,text/csv,application/csv,*/*\", retries=1, min_interval=0.75)\n                ctype = (rr.headers.get(\"content-type\") or \"\").lower()\n                body = rr.text.lstrip()\n\n                # csv=true is the authoritative tabular contract for the\n                # security-wise historical report. Sniff the body as well as\n                # the content type because some NSE responses are served with\n                # a generic application/json content type despite being CSV.\n                looks_like_csv = (\n                    params.get(\"csv\") == \"true\"\n                    or (\"csv\" in ctype)\n                    or (\n                        body\n                        and not body.startswith((\"{\", \"[\"))\n                        and \",\" in body[:500]\n                    )\n                )\n\n                if looks_like_csv:\n                    try:\n                        csv = pd.read_csv(\n                            io.BytesIO(rr.content),\n                            thousands=\",\",\n                        )\n                    except Exception:\n                        csv = pd.DataFrame()\n\n                if csv.empty:\n                    try:\n                        payload = rr.json()\n                        rows = payload.get(\n                            \"data\",\n                            payload if isinstance(payload, list) else [],\n                        )\n                        csv = pd.DataFrame(rows)\n                    except Exception:\n                        csv = pd.DataFrame()\n\n                if not csv.empty:\n                    r = rr\n                    break\n                r = rr\n            except Exception:\n                continue\n        if r is None:\n            raise RuntimeError(f\"NSE returned no response for {s} {cursor}..{chunk_end}\")\n        raw_dir = RAW_ROOT / \"equity\" / s\n        raw_dir.mkdir(parents=True, exist_ok=True)\n        raw_path = raw_dir / f\"{cursor:%Y%m%d}_{chunk_end:%Y%m%d}.csv\"\n        raw_path.write_bytes(r.content)\n        normalized = normalize_equity_history(csv, s)\n        if not normalized.empty:\n            frames.append(normalized)\n            out_path = NORMALIZED_ROOT / \"equity\" / s / f\"{cursor:%Y%m%d}_{chunk_end:%Y%m%d}.parquet\"\n            out_path.parent.mkdir(parents=True, exist_ok=True)\n            normalized.to_parquet(out_path, index=False)\n            if sync_drive:\n                _drive_upload(raw_path, f\"production_data/nse_official/raw/equity/{s}/{raw_path.name}\", \"text/csv\")\n                _drive_upload(out_path, f\"production_data/nse_official/equity/{s}/{out_path.name}\", \"application/octet-stream\")\n            m = _manifest()\n            key = f\"equity:{s}:{cursor.isoformat()}:{chunk_end.isoformat()}\"\n            m[\"records\"][key] = {\n                \"source\": \"NSE_OFFICIAL\",\n                \"symbol\": s,\n                \"from\": cursor.isoformat(), \"to\": chunk_end.isoformat(),\n                \"raw_local\": str(raw_path), \"normalized_local\": str(out_path),\n                \"rows\": int(len(normalized)), \"retrieved_at\": datetime.now(tz=IST).isoformat(),\n            }\n            _save_manifest(m)\n        cursor = chunk_end + timedelta(days=1)\n    if frames:\n        out = pd.concat(frames, ignore_index=True)\n        return (\n            out.sort_values(\"date\")\n            .drop_duplicates([\"symbol\", \"date\"], keep=\"last\")\n            .reset_index(drop=True)\n        )\n\n    # Direct NSE security-wise history can return an HTTP 200 JSON response\n    # without usable historical rows on hosted/cloud IPs. The project already\n    # maintains the same official NSE CM-UDiFF daily archive in Google Drive;\n    # use it as the persistent NSE-first fallback instead of jumping straight\n    # to a third-party provider.\n    try:\n        drive_history = fetch_nse_drive_bulk_symbol_history(\n            s,\n            start_d,\n            end_d,\n            max_calendar_days=240,\n        )\n        if not drive_history.empty:\n            return drive_history\n    except Exception:\n        pass\n\n    return pd.DataFrame()\n\n\n\nINDEX_HISTORY_ENDPOINT = \"/api/historicalOR/indicesHistory\"\nINDEX_HISTORY_NAMES = {\n    \"^NSEI\": \"NIFTY 50\",\n    \"^NSEBANK\": \"NIFTY BANK\",\n}\nINDEX_NORMALIZED_ROOT = NORMALIZED_ROOT / \"indices\"\n\n\ndef _normalize_index_history(frame: pd.DataFrame, requested_index: str) -> pd.DataFrame:\n    \"\"\"Normalize official NSE historical-index JSON into stable daily OHLC.\"\"\"\n    if frame is None or frame.empty:\n        return pd.DataFrame()\n\n    x = frame.copy()\n    x.columns = [str(c).strip() for c in x.columns]\n    aliases = {\n        \"EOD_TIMESTAMP\": \"date\",\n        \"TIMESTAMP\": \"date\",\n        \"Date\": \"date\",\n        \"EOD_OPEN_INDEX_VAL\": \"open\",\n        \"EOD_HIGH_INDEX_VAL\": \"high\",\n        \"EOD_LOW_INDEX_VAL\": \"low\",\n        \"EOD_CLOSE_INDEX_VAL\": \"close\",\n        \"OPEN\": \"open\",\n        \"HIGH\": \"high\",\n        \"LOW\": \"low\",\n        \"CLOSE\": \"close\",\n    }\n    x = x.rename(columns={c: aliases[c] for c in x.columns if c in aliases})\n    if \"date\" not in x.columns or any(c not in x.columns for c in (\"open\", \"high\", \"low\", \"close\")):\n        return pd.DataFrame()\n\n    raw_dates = x[\"date\"].astype(str).str.strip()\n    parsed = pd.to_datetime(raw_dates, format=\"%d-%b-%Y\", errors=\"coerce\")\n    remaining = parsed.isna()\n    for fmt in (\"%d-%m-%Y\", \"%Y-%m-%d\", \"%d/%m/%Y\", \"%Y%m%d\"):\n        if not remaining.any():\n            break\n        candidate = pd.to_datetime(raw_dates.loc[remaining], format=fmt, errors=\"coerce\")\n        parsed.loc[remaining] = candidate\n        remaining = parsed.isna()\n    if remaining.any():\n        candidate = pd.to_datetime(x.loc[remaining, \"date\"], errors=\"coerce\")\n        parsed.loc[remaining] = candidate\n\n    x[\"date\"] = parsed\n    for c in (\"open\", \"high\", \"low\", \"close\"):\n        x[c] = pd.to_numeric(x[c].astype(str).str.replace(\",\", \"\", regex=False), errors=\"coerce\")\n    if \"volume\" not in x.columns:\n        x[\"volume\"] = 0.0\n    else:\n        x[\"volume\"] = pd.to_numeric(x[\"volume\"].astype(str).str.replace(\",\", \"\", regex=False), errors=\"coerce\")\n\n    x[\"symbol\"] = str(requested_index).upper()\n    x[\"source\"] = \"NSE_OFFICIAL_INDEX_HISTORY\"\n    x[\"provenance\"] = \"nseindia.com /api/historicalOR/indicesHistory\"\n    x = x.dropna(subset=[\"date\", \"open\", \"high\", \"low\", \"close\"])\n    return (\n        x[[\"symbol\",\"date\",\"open\",\"high\",\"low\",\"close\",\"volume\",\"source\",\"provenance\"]]\n        .sort_values(\"date\")\n        .drop_duplicates([\"symbol\",\"date\"], keep=\"last\")\n        .reset_index(drop=True)\n    )\n\n\ndef _drive_index_history(\n    index_symbol: str,\n    start: date,\n    end: date,\n) -> pd.DataFrame:\n    \"\"\"Read persisted official NSE benchmark parquet files from Google Drive.\"\"\"\n    root = str(index_symbol).strip().upper()\n\n    try:\n        from persistent_store import GoogleDriveStore\n        from googleapiclient.http import MediaIoBaseDownload\n\n        store = GoogleDriveStore()\n\n        production = store._find_child(\n            store.root_id,\n            \"production_data\",\n            \"application/vnd.google-apps.folder\",\n        )\n        if not production:\n            return pd.DataFrame()\n\n        nse_official = store._find_child(\n            production[\"id\"],\n            \"nse_official\",\n            \"application/vnd.google-apps.folder\",\n        )\n        if not nse_official:\n            return pd.DataFrame()\n\n        indices = store._find_child(\n            nse_official[\"id\"],\n            \"indices\",\n            \"application/vnd.google-apps.folder\",\n        )\n        if not indices:\n            return pd.DataFrame()\n\n        folder_name = root.replace(\"^\", \"\").replace(\"/\", \"_\")\n\n        index_folder = store._find_child(\n            indices[\"id\"],\n            folder_name,\n            \"application/vnd.google-apps.folder\",\n        )\n\n        if not index_folder:\n            return pd.DataFrame()\n\n        page_token = None\n        files = []\n\n        while True:\n\n            kwargs = {\n                \"q\": (\n                    f\"'{index_folder['id']}' in parents \"\n                    \"and name contains '.parquet' \"\n                    \"and trashed = false\"\n                ),\n                \"spaces\": \"drive\",\n                \"fields\": \"nextPageToken,files(id,name)\",\n                \"pageSize\": 100,\n                \"includeItemsFromAllDrives\": True,\n                \"supportsAllDrives\": True,\n            }\n\n            if page_token:\n                kwargs[\"pageToken\"] = page_token\n\n            response = store.service.files().list(\n                **kwargs\n            ).execute()\n\n            files.extend(\n                response.get(\n                    \"files\",\n                    [],\n                )\n            )\n\n            page_token = response.get(\n                \"nextPageToken\"\n            )\n\n            if not page_token:\n                break\n\n        if not files:\n            return pd.DataFrame()\n\n        local_root = (\n            NORMALIZED_ROOT\n            / \"indices\"\n            / folder_name\n        )\n\n        parts = []\n\n        for item in files:\n\n            file_id = str(\n                item.get(\"id\", \"\")\n            ).strip()\n\n            name = str(\n                item.get(\"name\", \"\")\n            ).strip()\n\n            if not file_id or not name:\n                continue\n\n            local_path = (\n                local_root\n                / name\n            )\n\n            try:\n\n                if (\n                    not local_path.exists()\n                    or local_path.stat().st_size == 0\n                ):\n\n                    local_path.parent.mkdir(\n                        parents=True,\n                        exist_ok=True,\n                    )\n\n                    request = (\n                        store.service.files()\n                        .get_media(\n                            fileId=file_id\n                        )\n                    )\n\n                    temp = local_path.with_suffix(\n                        \".tmp\"\n                    )\n\n                    with temp.open(\"wb\") as fh:\n\n                        downloader = MediaIoBaseDownload(\n                            fh,\n                            request,\n                        )\n\n                        done = False\n\n                        while not done:\n                            _, done = downloader.next_chunk()\n\n                    temp.replace(\n                        local_path\n                    )\n\n                frame = pd.read_parquet(\n                    local_path\n                )\n\n                normalized = _normalize_index_history(\n                    frame,\n                    root,\n                )\n\n                if normalized.empty:\n                    # Some persisted files may already be normalized but carry\n                    # a different column spelling. Try the frame directly.\n                    frame = frame.copy()\n                    frame.columns = [\n                        str(col).strip().lower()\n                        for col in frame.columns\n                    ]\n\n                    if all(\n                        col in frame.columns\n                        for col in (\n                            \"date\",\n                            \"open\",\n                            \"high\",\n                            \"low\",\n                            \"close\",\n                        )\n                    ):\n                        for col in (\n                            \"open\",\n                            \"high\",\n                            \"low\",\n                            \"close\",\n                        ):\n                            frame[col] = pd.to_numeric(\n                                frame[col],\n                                errors=\"coerce\",\n                            )\n\n                        frame[\"date\"] = pd.to_datetime(\n                            frame[\"date\"],\n                            errors=\"coerce\",\n                        ).dt.normalize()\n\n                        frame[\"symbol\"] = root\n                        normalized = frame\n\n                if normalized.empty:\n                    continue\n\n                normalized[\"date\"] = pd.to_datetime(\n                    normalized[\"date\"],\n                    errors=\"coerce\",\n                ).dt.normalize()\n\n                normalized = normalized.dropna(\n                    subset=[\"date\"]\n                )\n\n                normalized = normalized[\n                    (normalized[\"date\"] >= pd.Timestamp(start))\n                    & (normalized[\"date\"] <= pd.Timestamp(end))\n                ]\n\n                if not normalized.empty:\n                    parts.append(\n                        normalized\n                    )\n\n            except Exception:\n                continue\n\n        if not parts:\n            return pd.DataFrame()\n\n        out = pd.concat(\n            parts,\n            ignore_index=True,\n        )\n\n        out[\"symbol\"] = root\n\n        return (\n            out\n            .sort_values(\"date\")\n            .drop_duplicates(\n                [\"symbol\", \"date\"],\n                keep=\"last\",\n            )\n            .reset_index(drop=True)\n        )\n\n    except Exception:\n        return pd.DataFrame()\n\n\ndef _fetch_index_history_nse_api(index_symbol: str, start: date | datetime, end: date | datetime,\n                        sync_drive: bool = False) -> pd.DataFrame:\n    \"\"\"Fetch/cache official NSE daily index history with bounded date chunks.\n\n    NSE's public historical-index endpoint can return a silently truncated\n    window for a large date range. To avoid treating partial history as valid,\n    requests are split into short chunks and the combined result is coverage-\n    checked against the requested range. Existing full-range caches are reused\n    only when they cover both ends of the requested period.\n    \"\"\"\n    root = str(index_symbol).strip().upper()\n    if root not in INDEX_HISTORY_NAMES:\n        raise ValueError(f\"Unsupported NSE benchmark: {index_symbol}\")\n\n    start_d = pd.Timestamp(start).date()\n    end_d = pd.Timestamp(end).date()\n    if end_d < start_d:\n        raise ValueError(\"end date cannot be before start date\")\n\n    cache_dir = INDEX_NORMALIZED_ROOT / root.replace(\"^\", \"\").replace(\"/\", \"_\")\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    cache_path = cache_dir / f\"{start_d:%Y%m%d}_{end_d:%Y%m%d}.parquet\"\n\n    def _coverage_ok(frame: pd.DataFrame, requested_start: date, requested_end: date) -> bool:\n        if frame is None or frame.empty or \"date\" not in frame.columns:\n            return False\n        dates = pd.to_datetime(frame[\"date\"], errors=\"coerce\").dropna()\n        if dates.empty:\n            return False\n        first = dates.min().date()\n        last = dates.max().date()\n        # Allow weekends/holidays around the requested boundaries, but reject\n        # materially truncated history such as a one-month response to a\n        # one-year request.\n        start_slack = 10\n        end_slack = 10\n        if first > requested_start + timedelta(days=start_slack):\n            return False\n        if last < requested_end - timedelta(days=end_slack):\n            return False\n        span_days = max(1, (requested_end - requested_start).days)\n        if span_days >= 90:\n            expected_weekdays = int(span_days * 5 / 7)\n            if len(dates) < max(30, int(expected_weekdays * 0.50)):\n                return False\n        return True\n\n    if cache_path.exists() and cache_path.stat().st_size > 0:\n        try:\n            cached = pd.read_parquet(cache_path)\n            if _coverage_ok(cached, start_d, end_d):\n                return cached\n        except Exception:\n            pass\n\n    frames: list[pd.DataFrame] = []\n    cursor = start_d\n    chunk_days = 31\n\n    while cursor <= end_d:\n        chunk_end = min(end_d, cursor + timedelta(days=chunk_days - 1))\n        params = {\n            \"indexType\": INDEX_HISTORY_NAMES[root],\n            \"from\": cursor.strftime(\"%d-%m-%Y\"),\n            \"to\": chunk_end.strftime(\"%d-%m-%Y\"),\n        }\n\n        response = _request(\n            INDEX_HISTORY_ENDPOINT,\n            params=params,\n            accept=\"application/json,text/plain,*/*\",\n            retries=2,\n            min_interval=0.75,\n        )\n\n        try:\n            payload = response.json()\n        except ValueError as exc:\n            raise RuntimeError(\n                f\"NSE historical index response was not valid JSON for \"\n                f\"{INDEX_HISTORY_NAMES[root]} {cursor}..{chunk_end}\"\n            ) from exc\n\n        rows = payload.get(\"data\", payload if isinstance(payload, list) else [])\n        normalized = _normalize_index_history(pd.DataFrame(rows), root)\n\n        if normalized.empty:\n            raise RuntimeError(\n                f\"NSE historical index request returned no usable rows for \"\n                f\"{INDEX_HISTORY_NAMES[root]} {cursor}..{chunk_end}\"\n            )\n\n        # A chunk may legitimately begin/end on a weekend or holiday. Reject\n        # only material truncation inside the requested chunk.\n        chunk_dates = pd.to_datetime(normalized[\"date\"], errors=\"coerce\").dropna()\n        if chunk_dates.empty:\n            raise RuntimeError(\n                f\"NSE historical index chunk contained no usable dates for \"\n                f\"{INDEX_HISTORY_NAMES[root]} {cursor}..{chunk_end}\"\n            )\n        if chunk_dates.min().date() > cursor + timedelta(days=10):\n            raise RuntimeError(\n                f\"NSE historical index chunk starts too late for \"\n                f\"{INDEX_HISTORY_NAMES[root]} {cursor}..{chunk_end}: \"\n                f\"{chunk_dates.min().date()}\"\n            )\n        if chunk_dates.max().date() < chunk_end - timedelta(days=10):\n            raise RuntimeError(\n                f\"NSE historical index chunk ends too early for \"\n                f\"{INDEX_HISTORY_NAMES[root]} {cursor}..{chunk_end}: \"\n                f\"{chunk_dates.max().date()}\"\n            )\n\n        frames.append(normalized)\n        cursor = chunk_end + timedelta(days=1)\n\n    combined = (\n        pd.concat(frames, ignore_index=True)\n        .sort_values(\"date\")\n        .drop_duplicates([\"symbol\", \"date\"], keep=\"last\")\n        .reset_index(drop=True)\n    )\n\n    if not _coverage_ok(combined, start_d, end_d):\n        raise RuntimeError(\n            f\"NSE historical index coverage incomplete for \"\n            f\"{INDEX_HISTORY_NAMES[root]}: \"\n            f\"requested={start_d}..{end_d}, \"\n            f\"actual={combined['date'].min().date()}..{combined['date'].max().date()}, \"\n            f\"rows={len(combined)}\"\n        )\n\n    combined.to_parquet(cache_path, index=False)\n\n    if sync_drive:\n        _drive_upload(\n            cache_path,\n            f\"production_data/nse_official/indices/{root.replace('^','')}/{cache_path.name}\",\n            \"application/octet-stream\",\n        )\n\n    return combined\n\n\n\ndef fetch_index_history(\n    index_symbol: str,\n    start: date | datetime,\n    end: date | datetime,\n    sync_drive: bool = False,\n) -> pd.DataFrame:\n    \"\"\"Fetch official NSE benchmark history with persistent Drive-first fallback.\n\n    Historical replay should not fail merely because the NSE index-history API\n    returns an HTML/non-JSON response from a hosted IP. The project already\n    persists official NIFTY 50 / NIFTY BANK history in Google Drive when those\n    files were previously retrieved successfully.\n    \"\"\"\n    root = str(index_symbol).strip().upper()\n\n    if root not in INDEX_HISTORY_NAMES:\n        raise ValueError(\n            f\"Unsupported NSE benchmark: {index_symbol}\"\n        )\n\n    start_d = pd.Timestamp(start).date()\n    end_d = pd.Timestamp(end).date()\n\n    # 1) Prefer already-persisted official NSE benchmark history.\n    try:\n        drive_frame = _drive_index_history(\n            root,\n            start_d,\n            end_d,\n        )\n\n        if (\n            drive_frame is not None\n            and not drive_frame.empty\n        ):\n            return drive_frame\n\n    except Exception:\n        pass\n\n    # 2) Use the existing official NSE API implementation.\n    try:\n        return _fetch_index_history_nse_api(\n            root,\n            start_d,\n            end_d,\n            sync_drive=sync_drive,\n        )\n\n    except Exception as api_exc:\n\n        # 3) One final Drive retry after an API failure. This is deliberately\n        # non-invasive and still uses only official persisted data.\n        try:\n            drive_frame = _drive_index_history(\n                root,\n                start_d,\n                end_d,\n            )\n\n            if (\n                drive_frame is not None\n                and not drive_frame.empty\n            ):\n                return drive_frame\n\n        except Exception:\n            pass\n\n        raise api_exc\n\n\n\n\nNSE_CM_UDIFF_URL_TEMPLATE = (\n    \"https://nsearchives.nseindia.com/content/cm/\"\n    \"BhavCopy_NSE_CM_0_0_0_{date}_F_0000.csv.zip\"\n)\nBULK_EOD_ROOT = NORMALIZED_ROOT / \"bulk_eod\"\nBULK_EOD_MANIFEST = BULK_EOD_ROOT / \"manifest.json\"\n\n\ndef _parse_nse_csv_bytes(payload: bytes) -> pd.DataFrame:\n    \"\"\"Read a CSV payload while tolerating UTF-8/BOM and alternate delimiters.\"\"\"\n    for encoding in (\"utf-8-sig\", \"latin-1\"):\n        try:\n            frame = pd.read_csv(io.BytesIO(payload), encoding=encoding)\n            if len(frame.columns) > 1:\n                return frame\n        except Exception:\n            continue\n    return pd.DataFrame()\n\n\ndef _normalize_udiff_common_bhavcopy(frame: pd.DataFrame, trade_date: date) -> pd.DataFrame:\n    if frame is None or frame.empty:\n        return pd.DataFrame()\n    x = frame.copy()\n    x.columns = [str(c).strip() for c in x.columns]\n    aliases = {\n        \"TradDt\": \"date\", \"TRADING_DATE\": \"date\", \"Date\": \"date\",\n        \"TckrSymb\": \"symbol\", \"CH_SYMBOL\": \"symbol\", \"Symbol\": \"symbol\",\n        \"SctySrs\": \"series\", \"CH_SERIES\": \"series\", \"Series\": \"series\",\n        \"OpnPric\": \"open\", \"CH_OPENING_PRICE\": \"open\", \"Open\": \"open\",\n        \"HghPric\": \"high\", \"CH_TRADE_HIGH_PRICE\": \"high\", \"High\": \"high\",\n        \"LwPric\": \"low\", \"CH_TRADE_LOW_PRICE\": \"low\", \"Low\": \"low\",\n        \"ClsPric\": \"close\", \"CH_CLOSING_PRICE\": \"close\", \"Close\": \"close\",\n        \"LastPric\": \"ltp\", \"CH_LAST_TRADED_PRICE\": \"ltp\", \"Last\": \"ltp\",\n        \"PrvsClsgPric\": \"prev_close\", \"CH_PREVIOUS_CLS_PRICE\": \"prev_close\",\n        \"TtlTradgVol\": \"volume\", \"CH_TOT_TRADED_QTY\": \"volume\", \"Volume\": \"volume\",\n        \"TtlTrfVal\": \"traded_value\", \"CH_TOT_TRADED_VAL\": \"traded_value\",\n        \"TtlNbOfTxsExctd\": \"num_trades\", \"CH_TOTAL_TRADES\": \"num_trades\",\n    }\n    ren = {c: aliases[c] for c in x.columns if c in aliases}\n    x = x.rename(columns=ren)\n    if \"date\" not in x.columns:\n        x[\"date\"] = pd.Timestamp(trade_date)\n\n    # NSE CM-UDiFF commonly supplies ISO dates such as YYYY-MM-DD.\n    # Do not parse these with dayfirst=True: that can silently transform\n    # 2026-07-01 into 2026-01-07 and similar month/day inversions.\n    raw_dates = x[\"date\"].astype(str).str.strip()\n    parsed_dates = pd.to_datetime(\n        raw_dates,\n        format=\"%Y-%m-%d\",\n        errors=\"coerce\",\n    )\n\n    # Support the other date representations seen in NSE extracts without\n    # applying dayfirst parsing to already-ISO values.\n    remaining = parsed_dates.isna()\n    for fmt in (\"%d-%b-%Y\", \"%d-%B-%Y\", \"%d-%m-%Y\", \"%d/%m/%Y\", \"%d-%b-%y\", \"%d/%b/%Y\", \"%Y%m%d\"):\n        if not remaining.any():\n            break\n        candidate = pd.to_datetime(\n            raw_dates.loc[remaining],\n            format=fmt,\n            errors=\"coerce\",\n        )\n        parsed_dates.loc[remaining] = candidate\n        remaining = parsed_dates.isna()\n\n    # Final fallback for datetime-like values that may already have been\n    # parsed by pandas before normalization.\n    if remaining.any():\n        candidate = pd.to_datetime(\n            x.loc[remaining, \"date\"],\n            errors=\"coerce\",\n        )\n        parsed_dates.loc[remaining] = candidate\n\n    x[\"date\"] = parsed_dates\n    if x[\"date\"].isna().all():\n        x[\"date\"] = pd.Timestamp(trade_date)\n    if \"symbol\" not in x.columns:\n        return pd.DataFrame()\n    x[\"symbol\"] = x[\"symbol\"].astype(str).str.strip().str.upper()\n    if \"series\" in x.columns:\n        x[\"series\"] = x[\"series\"].astype(str).str.strip().str.upper()\n    else:\n        x[\"series\"] = \"EQ\"\n    # Retain equity-like series; exclude ETFs/ETNs/debt from the stock universe.\n    x = x[x[\"series\"].isin({\"EQ\", \"BE\", \"BZ\", \"SM\", \"ST\", \"SZ\", \"GB\", \"RR\"})].copy()\n    for c in (\"open\", \"high\", \"low\", \"close\", \"ltp\", \"prev_close\", \"volume\", \"traded_value\", \"num_trades\"):\n        if c in x.columns:\n            x[c] = pd.to_numeric(x[c].astype(str).str.replace(\",\", \"\", regex=False), errors=\"coerce\")\n    if \"close\" not in x.columns and \"ltp\" in x.columns:\n        x[\"close\"] = x[\"ltp\"]\n    required = {\"symbol\", \"date\", \"open\", \"high\", \"low\", \"close\"}\n    if not required.issubset(x.columns):\n        return pd.DataFrame()\n    x[\"source\"] = \"NSE_OFFICIAL_UDIFF_BHAVCOPY\"\n    x[\"provenance\"] = \"nsearchives.nseindia.com CM-UDiFF Common Bhavcopy\"\n    return x.sort_values([\"symbol\", \"date\"]).drop_duplicates([\"symbol\", \"date\"], keep=\"last\").reset_index(drop=True)\n\n\ndef fetch_nse_eod_bhavcopy(trade_date: date | datetime, sync_drive: bool = False,\n                           force_refresh: bool = False) -> pd.DataFrame:\n    \"\"\"Fetch one official NSE CM-UDiFF common bhavcopy containing all equity symbols.\n\n    This is the bulk-EOD path used to expand coverage without issuing one NSE\n    request per security. The downloaded day is cached locally and can be\n    mirrored to Google Drive when explicitly requested.\n    \"\"\"\n    global _last_nse_diagnostic\n\n    d = pd.Timestamp(trade_date).date()\n    key = d.strftime(\"%Y%m%d\")\n    out_dir = BULK_EOD_ROOT / f\"{d:%Y}\" / f\"{d:%m}\" / f\"{d:%d}\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    parquet_path = out_dir / f\"BhavCopy_NSE_CM_0_0_0_{key}_F_0000.parquet\"\n    zip_path = out_dir / f\"BhavCopy_NSE_CM_0_0_0_{key}_F_0000.csv.zip\"\n    if parquet_path.exists() and parquet_path.stat().st_size > 0 and not force_refresh:\n        return pd.read_parquet(parquet_path)\n    url = NSE_CM_UDIFF_URL_TEMPLATE.format(date=key)\n    # Archive endpoint is a direct static download; use one bounded attempt.\n    _sleep_rate_limit(0.25)\n    try:\n        r = requests.get(\n            url,\n            headers={**HEADERS, \"Accept\": \"application/zip,text/plain,*/*\"},\n            timeout=(3, 8),\n        )\n        _last_nse_diagnostic = {\n            \"path\": url,\n            \"status_code\": r.status_code,\n            \"content_type\": r.headers.get(\"content-type\", \"\"),\n            \"bytes\": len(r.content),\n        }\n        r.raise_for_status()\n    except requests.HTTPError as exc:\n        response = getattr(exc, \"response\", None)\n        status_code = getattr(response, \"status_code\", None)\n        _last_nse_diagnostic.update(\n            {\n                \"path\": url,\n                \"status_code\": status_code,\n                \"error\": f\"{type(exc).__name__}: {exc}\",\n            }\n        )\n        raise RuntimeError(\n            f\"NSE EOD bhavcopy unavailable for {key}: \"\n            f\"{type(exc).__name__}: {exc}\"\n        ) from exc\n    except requests.RequestException as exc:\n        _last_nse_diagnostic.update(\n            {\n                \"path\": url,\n                \"status_code\": None,\n                \"error\": f\"{type(exc).__name__}: {exc}\",\n            }\n        )\n        raise RuntimeError(\n            f\"NSE EOD bhavcopy unavailable for {key}: \"\n            f\"{type(exc).__name__}: {exc}\"\n        ) from exc\n    zip_path.write_bytes(r.content)\n    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:\n        names = [n for n in zf.namelist() if not n.endswith(\"/\")]\n        if not names:\n            return pd.DataFrame()\n        # Prefer the main CM CSV, otherwise first CSV-like member.\n        name = next((n for n in names if n.lower().endswith(\".csv\")), names[0])\n        raw = zf.read(name)\n    frame = _parse_nse_csv_bytes(raw)\n    normalized = _normalize_udiff_common_bhavcopy(frame, d)\n    if normalized.empty:\n        raise RuntimeError(f\"NSE EOD bhavcopy parsed but yielded no equity rows for {key}\")\n    normalized.to_parquet(parquet_path, index=False)\n    if sync_drive:\n        _drive_upload(zip_path, f\"production_data/nse_official/bulk_eod/{d:%Y/%m/%d}/{zip_path.name}\", \"application/zip\")\n        _drive_upload(parquet_path, f\"production_data/nse_official/bulk_eod/{d:%Y/%m/%d}/{parquet_path.name}\", \"application/octet-stream\")\n    if BULK_EOD_MANIFEST.exists():\n        try:\n            m = json.loads(BULK_EOD_MANIFEST.read_text())\n            if not isinstance(m, dict):\n                raise ValueError(\"bulk EOD manifest is not a JSON object\")\n        except (OSError, ValueError, json.JSONDecodeError):\n            # A previously interrupted runner can leave a partial manifest.\n            # Reinitialize it from the files already present on disk.\n            m = {\"version\": \"nse-cm-udiff-v1\", \"days\": {}}\n    else:\n        m = {\"version\": \"nse-cm-udiff-v1\", \"days\": {}}\n    m.setdefault(\"days\", {})[key] = {\n        \"source\": \"NSE_OFFICIAL_UDIFF_BHAVCOPY\", \"url\": url,\n        \"parquet\": str(parquet_path), \"rows\": int(len(normalized)),\n        \"retrieved_at\": datetime.now(tz=IST).isoformat(),\n    }\n    BULK_EOD_MANIFEST.write_text(json.dumps(m, indent=2, sort_keys=True, default=str))\n    return normalized\n\n\ndef _drive_bulk_file_index() -> dict[str, str]:\n    \"\"\"Return a cached Google Drive index of official NSE bulk-EOD parquet files.\"\"\"\n    cache = getattr(_drive_bulk_file_index, \"_cache\", None)\n    if cache is not None:\n        return cache\n\n    try:\n        from persistent_store import GoogleDriveStore\n\n        store = GoogleDriveStore()\n\n        files = {}\n        token = None\n\n        while True:\n            kwargs = {\n                \"q\": (\n                    \"name contains 'BhavCopy_NSE_CM_0_0_0_' \"\n                    \"and name contains '.parquet' and trashed = false\"\n                ),\n                \"spaces\": \"drive\",\n                \"fields\": \"nextPageToken,files(id,name)\",\n                \"pageSize\": 1000,\n                \"includeItemsFromAllDrives\": True,\n                \"supportsAllDrives\": True,\n            }\n\n            if token:\n                kwargs[\"pageToken\"] = token\n\n            resp = store.service.files().list(**kwargs).execute()\n\n            for item in resp.get(\"files\", []):\n                name = str(item.get(\"name\", \"\")).strip()\n                file_id = str(item.get(\"id\", \"\")).strip()\n                if name and file_id:\n                    files[name] = file_id\n\n            token = resp.get(\"nextPageToken\")\n\n            if not token:\n                break\n\n        _drive_bulk_file_index._cache = files\n        return files\n\n    except Exception:\n        _drive_bulk_file_index._cache = {}\n        return {}\n\n\ndef _drive_download_bulk_day(filename: str, destination: Path) -> bool:\n    \"\"\"Download one official NSE bulk-EOD parquet from Google Drive.\"\"\"\n    try:\n        from persistent_store import GoogleDriveStore\n        from googleapiclient.http import MediaIoBaseDownload\n\n        index = _drive_bulk_file_index()\n        file_id = index.get(filename)\n\n        if not file_id:\n            return False\n\n        store = GoogleDriveStore()\n\n        request = store.service.files().get_media(\n            fileId=file_id\n        )\n\n        destination.parent.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n\n        temp = destination.with_suffix(\n            destination.suffix + \".tmp\"\n        )\n\n        with temp.open(\"wb\") as fh:\n            downloader = MediaIoBaseDownload(\n                fh,\n                request,\n            )\n\n            done = False\n\n            while not done:\n                _, done = downloader.next_chunk()\n\n        temp.replace(destination)\n        return True\n\n    except Exception:\n        return False\n\n\ndef fetch_nse_drive_bulk_symbol_history(\n    symbol: str,\n    start: date | datetime,\n    end: date | datetime,\n    max_calendar_days: int = 240,\n) -> pd.DataFrame:\n    \"\"\"Read official NSE bulk-EOD history from Google Drive when direct NSE\n    security-history HTTP is unavailable.\n\n    The Drive archive is the persistent official NSE source already populated\n    by the project's EOD ingestion workflow. Only the requested symbol is\n    retained from each day, and downloaded day files are cached locally.\n    \"\"\"\n    s = root_symbol(symbol)\n\n    start_d = pd.Timestamp(start).date()\n    end_d = pd.Timestamp(end).date()\n\n    earliest = end_d - timedelta(\n        days=int(max_calendar_days)\n    )\n\n    if start_d < earliest:\n        start_d = earliest\n\n    days = pd.date_range(\n        start_d,\n        end_d,\n        freq=\"B\",\n    )\n\n    if len(days) == 0:\n        return pd.DataFrame()\n\n    pieces = []\n\n    for day in days:\n\n        d = day.date()\n\n        filename = (\n            f\"BhavCopy_NSE_CM_0_0_0_\"\n            f\"{d:%Y%m%d}_F_0000.parquet\"\n        )\n\n        local_path = (\n            BULK_EOD_ROOT\n            / f\"{d:%Y}\"\n            / f\"{d:%m}\"\n            / f\"{d:%d}\"\n            / filename\n        )\n\n        try:\n            if local_path.exists() and local_path.stat().st_size > 0:\n                day_df = pd.read_parquet(\n                    local_path,\n                    columns=None,\n                )\n\n            else:\n                ok = _drive_download_bulk_day(\n                    filename,\n                    local_path,\n                )\n\n                if not ok:\n                    continue\n\n                day_df = pd.read_parquet(\n                    local_path,\n                    columns=None,\n                )\n\n            if day_df.empty:\n                continue\n\n            if \"symbol\" not in day_df.columns:\n                continue\n\n            part = day_df[\n                day_df[\"symbol\"]\n                .astype(str)\n                .str.strip()\n                .str.upper()\n                == s\n            ].copy()\n\n            if not part.empty:\n                pieces.append(part)\n\n        except Exception:\n            continue\n\n    if not pieces:\n        return pd.DataFrame()\n\n    out = pd.concat(\n        pieces,\n        ignore_index=True,\n    )\n\n    out[\"symbol\"] = (\n        out[\"symbol\"]\n        .astype(str)\n        .str.strip()\n        .str.upper()\n    )\n\n    out[\"date\"] = pd.to_datetime(\n        out[\"date\"],\n        errors=\"coerce\",\n    ).dt.normalize()\n\n    return (\n        out\n        .dropna(\n            subset=[\n                \"symbol\",\n                \"date\",\n            ]\n        )\n        .sort_values(\"date\")\n        .drop_duplicates(\n            [\n                \"symbol\",\n                \"date\",\n            ],\n            keep=\"last\",\n        )\n        .reset_index(drop=True)\n    )\n\n\ndef fetch_nse_bulk_symbol_history(symbol: str, start: date | datetime, end: date | datetime,\n                                  sync_drive: bool = False, max_days: int | None = None,\n                                  download_missing: bool = False) -> pd.DataFrame:\n    \"\"\"Build per-symbol daily history from official NSE bulk CM-UDiFF day files.\n\n    Cached days are read locally; missing days are downloaded one at a time. The\n    function is intended for scheduled/backfill jobs, not tight UI loops.\n    \"\"\"\n    s = root_symbol(symbol)\n    start_d = pd.Timestamp(start).date(); end_d = pd.Timestamp(end).date()\n    days = pd.date_range(start_d, end_d, freq=\"B\")\n    if max_days is not None:\n        days = days[-int(max_days):]\n    pieces = []\n    for day in days:\n        try:\n            d = day.date()\n            cache = BULK_EOD_ROOT / f\"{d:%Y}\" / f\"{d:%m}\" / f\"{d:%d}\" / f\"BhavCopy_NSE_CM_0_0_0_{d:%Y%m%d}_F_0000.parquet\"\n            if cache.exists():\n                df = pd.read_parquet(cache)\n            elif download_missing:\n                df = fetch_nse_eod_bhavcopy(d, sync_drive=sync_drive)\n            else:\n                continue\n            part = df[df[\"symbol\"].astype(str).str.upper() == s]\n            if not part.empty:\n                pieces.append(part)\n        except Exception:\n            continue\n    if not pieces:\n        return pd.DataFrame()\n    out = pd.concat(pieces, ignore_index=True)\n    return out.sort_values(\"date\").drop_duplicates([\"symbol\", \"date\"], keep=\"last\").reset_index(drop=True)\n\nEQUITY_UNIVERSE_URL = \"https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv\"\n\ndef fetch_nse_equity_universe(force_refresh: bool = False) -> pd.DataFrame:\n    \"\"\"Fetch the current NSE-listed/available-for-trading equity universe.\n\n    This is the official NSE securities CSV linked from the current\n    \"Securities available for Trading\" page.\n    \"\"\"\n    cache = NORMALIZED_ROOT / \"universe\" / \"EQUITY_L.csv\"\n    if cache.exists() and not force_refresh and (time.time() - cache.stat().st_mtime) < 86400:\n        return pd.read_csv(cache)\n    _ensure_local()\n    cache.parent.mkdir(parents=True, exist_ok=True)\n    try:\n        r = requests.get(EQUITY_UNIVERSE_URL, headers=HEADERS, timeout=(3, 8))\n        r.raise_for_status()\n        cache.write_bytes(r.content)\n        df = pd.read_csv(io.BytesIO(r.content))\n    except Exception:\n        if cache.exists():\n            return pd.read_csv(cache)\n        raise\n    return df\n\ndef nse_equity_symbols(force_refresh: bool = False) -> list[str]:\n    df = fetch_nse_equity_universe(force_refresh=force_refresh)\n    col = next((c for c in df.columns if str(c).strip().upper() in {\"SYMBOL\", \"SYMBOL_NAME\", \"CH_SYMBOL\"}), None)\n    if col is None:\n        raise ValueError(f\"NSE equity universe has no symbol column: {list(df.columns)}\")\n    vals = (df[col].astype(str).str.strip().str.upper()\n            .replace({\"NAN\": \"\", \"NONE\": \"\"}))\n    return sorted({v for v in vals if v and v not in {\"SYMBOL\", \"SCRIPCODE\"}})\n\ndef fetch_quote(symbol: str, trade_info: bool = False) -> dict[str, Any]:\n    s = root_symbol(symbol)\n    params = {\"symbol\": s}\n    if trade_info:\n        params[\"section\"] = \"trade_info\"\n    r = _request(EQUITY_QUOTE_ENDPOINT, params=params)\n    return r.json()\n\n\ndef fetch_corporate_actions(symbol: str) -> list[dict[str, Any]]:\n    s = root_symbol(symbol)\n    r = _request(CORPORATE_ACTIONS_ENDPOINT, params={\"index\": \"equities\", \"symbol\": s})\n    data = r.json()\n    return data if isinstance(data, list) else data.get(\"data\", []) if isinstance(data, dict) else []\n\n\ndef fetch_corporate_announcements(symbol: str) -> list[dict[str, Any]]:\n    s = root_symbol(symbol)\n    r = _request(CORPORATE_ANNOUNCEMENTS_ENDPOINT, params={\"index\": \"equities\", \"symbol\": s})\n    data = r.json()\n    return data if isinstance(data, list) else data.get(\"data\", []) if isinstance(data, dict) else []\n\n\ndef fetch_intraday_1m_current(symbol: str, sync_drive: bool = True) -> pd.DataFrame:\n    \"\"\"Fetch today's NSE quote-chart path and aggregate available prints to 1-minute OHLC.\n\n    NSE's public quote chart is an LTP/time series, not a full broker websocket.\n    Volume is therefore left missing in this path; it must not be invented.\n    \"\"\"\n    s = root_symbol(symbol)\n    params = {\"index\": s + \"EQN\"}\n    r = _request(EQUITY_CHART_ENDPOINT, params=params)\n    payload = r.json()\n    pts = payload.get(\"grapthData\", []) if isinstance(payload, dict) else []\n    if not pts:\n        return pd.DataFrame()\n    frame = pd.DataFrame(pts, columns=[\"epoch_ms\", \"ltp\"])\n    frame[\"timestamp\"] = pd.to_datetime(frame[\"epoch_ms\"], unit=\"ms\", utc=True).dt.tz_convert(IST)\n    frame[\"ltp\"] = pd.to_numeric(frame[\"ltp\"], errors=\"coerce\")\n    frame = frame.dropna(subset=[\"timestamp\", \"ltp\"]).set_index(\"timestamp\").sort_index()\n    bars = frame[\"ltp\"].resample(\"1min\", label=\"left\", closed=\"left\").ohlc()\n    bars[\"Volume\"] = np.nan\n    bars = bars.rename(columns={\"open\": \"Open\", \"high\": \"High\", \"low\": \"Low\", \"close\": \"Close\"})\n    bars[\"symbol\"] = s\n    bars[\"source\"] = \"NSE_OFFICIAL_CHART\"\n    bars[\"provenance\"] = \"nseindia.com public equity quote chart\"\n    bars = bars.dropna(subset=[\"Open\", \"High\", \"Low\", \"Close\"])\n    if sync_drive and not bars.empty:\n        try:\n            from persistent_store import persist_1m_frame\n            persist_1m_frame(s, bars, source=\"NSE_OFFICIAL_CHART\", sync_drive=True)\n        except Exception:\n            pass\n    return bars\n\n\ndef fetch_fno_history(symbol: str, start: date | datetime, end: date | datetime,\n                      instrument_type: str, expiry_date: date | datetime,\n                      option_type: str | None = None, strike_price: float | None = None,\n                      sync_drive: bool = True) -> pd.DataFrame:\n    \"\"\"Fetch a contract's official NSE historical F&O data.\n\n    NSE currently exposes contract-wise historical data through the historicalOR\n    interface. The caller must specify the contract, expiry and (for options)\n    option type/strike to keep the data provenance unambiguous.\n    \"\"\"\n    s = root_symbol(symbol)\n    start_d, end_d = pd.Timestamp(start).date(), pd.Timestamp(end).date()\n    expiry = pd.Timestamp(expiry_date).strftime(\"%d-%b-%Y\")\n    frames = []\n    cursor = start_d\n    while cursor <= end_d:\n        chunk_end = min(cursor + timedelta(days=59), end_d)\n        params: dict[str, Any] = {\n            \"from\": cursor.strftime(\"%d-%m-%Y\"),\n            \"to\": chunk_end.strftime(\"%d-%m-%Y\"),\n            \"instrumentType\": instrument_type,\n            \"symbol\": s,\n            \"year\": str(pd.Timestamp(expiry_date).year),\n            \"expiryDate\": expiry,\n        }\n        if option_type:\n            params[\"optionType\"] = option_type\n        if strike_price is not None:\n            params[\"strikePrice\"] = str(float(strike_price))\n        r = _request(FNO_HISTORY_ENDPOINT, params=params)\n        payload = r.json()\n        data = payload.get(\"data\", payload if isinstance(payload, list) else [])\n        df = pd.DataFrame(data)\n        if not df.empty:\n            df[\"source\"] = \"NSE_OFFICIAL\"\n            df[\"provenance\"] = \"nseindia.com historical F&O contract interface\"\n            frames.append(df)\n            raw_dir = RAW_ROOT / \"fno\" / s\n            raw_dir.mkdir(parents=True, exist_ok=True)\n            out_path = NORMALIZED_ROOT / \"fno\" / s / f\"{cursor:%Y%m%d}_{chunk_end:%Y%m%d}_{instrument_type}_{pd.Timestamp(expiry_date):%Y%m%d}.parquet\"\n            out_path.parent.mkdir(parents=True, exist_ok=True)\n            df.to_parquet(out_path, index=False)\n            if sync_drive:\n                _drive_upload(out_path, f\"production_data/nse_official/fno/{s}/{out_path.name}\")\n        cursor = chunk_end + timedelta(days=1)\n    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()\n\n\ndef source_status() -> dict[str, Any]:\n    \"\"\"Non-blocking NSE status based on official universe archive reachability/cache.\n\n    Quote-equity is intentionally not used as a startup health gate because a\n    transient quote-endpoint block must not prevent historical/data-backed\n    predictions.\n    \"\"\"\n    try:\n        df = fetch_nse_equity_universe(force_refresh=False)\n        col = next((c for c in df.columns if str(c).strip().upper() in {\"SYMBOL\", \"CH_SYMBOL\"}), None)\n        count = int(df[col].astype(str).str.strip().nunique()) if col else 0\n        return {\"status\": \"PASS\" if count else \"ERROR\", \"source\": \"NSE_OFFICIAL\", \"endpoint\": EQUITY_UNIVERSE_URL, \"universe_symbols\": count, \"quote_health_check\": False}\n    except Exception as exc:\n        return {\"status\": \"ERROR\", \"source\": \"NSE_OFFICIAL\", \"endpoint\": EQUITY_UNIVERSE_URL, \"error\": f\"{type(exc).__name__}: {exc}\", \"quote_health_check\": False}\n","delivery_training_universe.py":"\"\"\"Training-universe policy for the next-session Delivery model.\n\nThe training universe is the current official NSE equity universe, not a hardcoded\nwatchlist. Historical observations still come from the persistent official NSE\nCM-UDiFF bulk EOD archive. A listed symbol is included in the training roster,\nbut it is only trainable when enough clean historical sessions are present.\n\nThis module does not alter the currently deployed production model. It defines\nthe universe that future candidate training runs must use.\n\"\"\"\nfrom __future__ import annotations\n\nfrom datetime import datetime\nfrom typing import Any\n\nimport pandas as pd\n\n\nEQUITY_SERIES = {\"EQ\", \"BE\", \"BZ\", \"SM\", \"ST\", \"SZ\", \"GB\", \"RR\"}\n\n\ndef normalize_symbol(value: Any) -> str:\n    s = str(value or \"\").strip().upper()\n    if s.endswith(\".NS\") or s.endswith(\".BO\"):\n        s = s.rsplit(\".\", 1)[0]\n    return s\n\n\ndef load_official_nse_universe(force_refresh: bool = False) -> pd.DataFrame:\n    \"\"\"Load the official current NSE securities-available-for-trading roster.\"\"\"\n    from nse_data_layer import fetch_nse_equity_universe\n\n    frame = fetch_nse_equity_universe(force_refresh=force_refresh).copy()\n    column = next(\n        (\n            c\n            for c in frame.columns\n            if str(c).strip().upper() in {\"SYMBOL\", \"CH_SYMBOL\", \"SYMBOL_NAME\"}\n        ),\n        None,\n    )\n    if column is None:\n        raise RuntimeError(\n            f\"NSE universe response has no symbol column: {list(frame.columns)}\"\n        )\n    frame[\"_symbol\"] = frame[column].map(normalize_symbol)\n    frame = frame[frame[\"_symbol\"].ne(\"\")].copy()\n\n    series_col = next(\n        (\n            c\n            for c in frame.columns\n            if str(c).strip().upper() in {\"SERIES\", \"CH_SERIES\"}\n        ),\n        None,\n    )\n    if series_col is not None:\n        frame[\"_series\"] = frame[series_col].astype(str).str.strip().str.upper()\n        frame = frame[\n            frame[\"_series\"].isin(EQUITY_SERIES) | frame[\"_series\"].eq(\"\")\n        ].copy()\n\n    return frame.drop_duplicates(\"_symbol\", keep=\"last\").reset_index(drop=True)\n\n\ndef official_nse_symbols(force_refresh: bool = False) -> list[str]:\n    return sorted(load_official_nse_universe(force_refresh=force_refresh)[\"_symbol\"].tolist())\n\n\ndef training_roster(bulk_frame: pd.DataFrame, min_history: int = 60) -> dict[str, Any]:\n    \"\"\"Return the full official roster plus trainable/excluded symbols.\n\n    The bulk_frame is expected to be the already-normalized official NSE bulk\n    EOD dataset. No public fallback data is substituted here.\n    \"\"\"\n    universe = set(official_nse_symbols(force_refresh=False))\n    bulk = bulk_frame.copy()\n    bulk[\"_symbol\"] = bulk[\"symbol\"].map(normalize_symbol)\n    counts = bulk.groupby(\"_symbol\")[\"date\"].nunique()\n\n    available = sorted(universe & set(counts.index))\n    trainable = sorted(s for s in available if int(counts.get(s, 0)) >= min_history)\n    excluded_set = set(universe) - set(trainable)\n    excluded = sorted(excluded_set)\n\n    return {\n        \"universe_source\": \"NSE_OFFICIAL_EQUITY_L\",\n        \"universe_as_of\": datetime.now().astimezone().isoformat(),\n        \"universe_symbol_count\": len(universe),\n        \"symbols_with_bulk_history\": len(available),\n        \"trainable_symbol_count\": len(trainable),\n        \"trainable_symbols\": trainable,\n        \"excluded_symbol_count\": len(excluded),\n        \"excluded_symbols\": excluded,\n        \"minimum_history_sessions\": int(min_history),\n    }\n\n\ndef benchmark_for_symbol(symbol: str) -> str:\n    root = normalize_symbol(symbol)\n    banking = {\n        \"HDFCBANK\", \"ICICIBANK\", \"SBIN\", \"AXISBANK\", \"KOTAKBANK\",\n        \"INDUSINDBK\", \"BANKBARODA\", \"PNB\",\n    }\n    return \"^NSEBANK\" if root in banking else \"^NSEI\"\n","multifactor_data_layer_v2.py":"\"\"\"Truthful point-in-time multi-factor readiness utilities.\n\nStandardizes availability flags for news, corporate events, F&O and\nfundamentals for the existing 59-feature ALL-NSE Delivery contract.\nMissing historical information is never fabricated.\n\"\"\"\nfrom __future__ import annotations\nfrom pathlib import Path\nfrom typing import Iterable\nimport numpy as np\nimport pandas as pd\n\nFEATURES_59 = [\n    \"ret_1\",\"ret_3\",\"ret_6\",\"ret_12\",\"ret_24\",\"ret_48\",\"rsi14\",\"atr14\",\n    \"ma20_ratio\",\"ma50_ratio\",\"volume_ratio\",\"volume_z20\",\"delivery_pct\",\n    \"delivery_pct_change_1\",\"delivery_pct_ratio20\",\"delivery_pct_z20\",\n    \"delivery_qty_ratio20\",\"delivery_qty_z20\",\"range_pct\",\"body_pct\",\"up_bar\",\n    \"market_ret_1\",\"market_ret_3\",\"market_ret_6\",\"market_rsi14\",\n    \"relative_ret_1\",\"relative_ret_6\",\"relative_ret_12\",\"market_mood_proxy\",\n    \"market_stress_proxy\",\"news_available\",\"event_available\",\"event_high_impact\",\n    \"fno_pcr\",\"fno_oi_change\",\"fno_iv_change\",\"fno_available\",\"fno_fut_close\",\n    \"fno_fut_basis\",\"fno_fut_return\",\"fno_fut_oi\",\"fundamental_revenue\",\"fundamental_pat\",\n    \"fundamental_eps\",\"fundamental_ebitda\",\"fundamental_finance_cost\",\n    \"fundamental_total_equity\",\"fundamental_total_debt\",\"fundamental_cash\",\n    \"fund_revenue_growth\",\"fund_pat_growth\",\"fund_eps_growth\",\n    \"fund_ebitda_growth\",\"fund_pat_margin\",\"fund_debt_to_equity\",\n    \"fund_cash_to_debt\",\"fundamental_age_days\",\"fundamental_stale_90d\",\n    \"fundamental_stale_180d\",\n]\n\nGROUPS = {\n    \"market_price\": FEATURES_59[:21],\n    \"market_context\": FEATURES_59[21:30],\n    \"news\": [\"news_available\"],\n    \"events\": [\"event_available\",\"event_high_impact\"],\n    \"fno\": FEATURES_59[33:41],\n    \"fundamentals\": FEATURES_59[41:],\n}\nif len(FEATURES_59) != 59:\n    raise RuntimeError(f\"Feature contract must contain exactly 59 features; found {len(FEATURES_59)}\")\n\nREADINESS_COLUMNS = {\n    \"market_price\":\"ready_market_price\",\n    \"market_context\":\"ready_market_context\",\n    \"news\":\"ready_news\",\n    \"events\":\"ready_events\",\n    \"fno\":\"ready_fno\",\n    \"fundamentals\":\"ready_fundamentals\",\n}\n\ndef normalize_symbol(value: object) -> str:\n    s = str(value or \"\").strip().upper()\n    for suffix in (\".NS\",\".BO\",\"-EQ\",\".EQ\"):\n        if s.endswith(suffix):\n            s=s[: -len(suffix)]\n    return s\n\ndef normalize_timestamp(frame: pd.DataFrame, preferred: Iterable[str] = ()) -> pd.DataFrame:\n    x=frame.copy()\n    if \"decision_timestamp\" not in x.columns:\n        aliases=list(preferred)+[\"timestamp\",\"datetime\",\"date_time\",\"bar_timestamp\",\"trade_timestamp\",\"available_at\",\"publication_timestamp\",\"observed_at\",\"fundamental_available_at\",\"fno_observed_at\"]\n        present=[c for c in aliases if c in x.columns]\n        if len(present)!=1:\n            raise ValueError(f\"Unable to resolve PIT timestamp; present={present}\")\n        x=x.rename(columns={present[0]:\"decision_timestamp\"})\n    x[\"decision_timestamp\"]=pd.to_datetime(x[\"decision_timestamp\"],errors=\"coerce\",utc=True)\n    if \"symbol\" not in x.columns:\n        raise ValueError(\"PIT frame requires symbol\")\n    x[\"symbol\"]=x[\"symbol\"].map(normalize_symbol)\n    return x.dropna(subset=[\"symbol\",\"decision_timestamp\"]).copy()\n\ndef _available_from_values(frame: pd.DataFrame, cols: list[str]) -> pd.Series:\n    present=[c for c in cols if c in frame.columns]\n    if not present:\n        return pd.Series(np.nan,index=frame.index,dtype=\"float64\")\n    numeric=frame[present].apply(pd.to_numeric,errors=\"coerce\")\n    return numeric.notna().any(axis=1).astype(\"Int8\")\n\ndef attach_readiness_flags(frame: pd.DataFrame) -> pd.DataFrame:\n    x=frame.copy()\n    for c in FEATURES_59:\n        if c not in x.columns:\n            x[c]=np.nan\n    x[\"ready_market_price\"]=_available_from_values(x,GROUPS[\"market_price\"])\n    x[\"ready_market_context\"]=_available_from_values(x,GROUPS[\"market_context\"])\n\n    for group,source in ((\"news\",\"news_available\"),(\"events\",\"event_available\"),(\"fno\",\"fno_available\")):\n        if source in x.columns:\n            s=pd.to_numeric(x[source],errors=\"coerce\")\n            x[READINESS_COLUMNS[group]]=s.astype(\"Int8\")\n        else:\n            x[READINESS_COLUMNS[group]]=_available_from_values(x,GROUPS[group])\n\n    if \"fundamentals_available\" in x.columns:\n        x[\"ready_fundamentals\"]=pd.to_numeric(x[\"fundamentals_available\"],errors=\"coerce\").astype(\"Int8\")\n    else:\n        x[\"ready_fundamentals\"]=_available_from_values(x,GROUPS[\"fundamentals\"])\n\n    flags=list(READINESS_COLUMNS.values())\n    x[\"information_groups_available\"]=x[flags].notna().sum(axis=1).astype(\"Int8\")\n    x[\"information_groups_ready\"]=x[flags].fillna(0).astype(float).sum(axis=1).astype(\"Int8\")\n    x[\"multi_factor_full_ready\"]=x[flags].eq(1).all(axis=1).astype(\"Int8\")\n    x[\"core_market_ready\"]=(x[\"ready_market_price\"].eq(1)&x[\"ready_market_context\"].eq(1)).astype(\"Int8\")\n    x[\"data_readiness_score\"]=x[flags].fillna(0).astype(float).sum(axis=1)/len(flags)\n    x[\"feature_non_null_fraction\"]=x[FEATURES_59].notna().mean(axis=1)\n    return x\n\ndef readiness_report(frame: pd.DataFrame) -> dict:\n    x=attach_readiness_flags(frame)\n    n=len(x)\n    out={\n        \"rows\":int(n),\n        \"symbols\":int(x[\"symbol\"].nunique()) if \"symbol\" in x else 0,\n        \"mean_group_readiness_pct\":round(float(x[\"data_readiness_score\"].mean()*100),3) if n else 0.0,\n        \"full_multifactor_ready_pct\":round(float(x[\"multi_factor_full_ready\"].mean()*100),3) if n else 0.0,\n        \"mean_feature_non_null_pct\":round(float(x[\"feature_non_null_fraction\"].mean()*100),3) if n else 0.0,\n        \"groups\":{},\n    }\n    for group,flag in READINESS_COLUMNS.items():\n        s=x[flag]\n        out[\"groups\"][group]={\n            \"known_rows\":int(s.notna().sum()),\n            \"ready_rows\":int(s.eq(1).sum()),\n            \"known_coverage_pct\":round(float(s.notna().mean()*100),3) if n else 0.0,\n            \"ready_coverage_pct\":round(float(s.eq(1).mean()*100),3) if n else 0.0,\n            \"ready_symbols\":int(x.loc[s.eq(1),\"symbol\"].nunique()) if n else 0,\n        }\n    return out\n\ndef merge_pit_layer(base: pd.DataFrame, layer: pd.DataFrame, value_columns: list[str], source_name: str) -> pd.DataFrame:\n    if layer is None or layer.empty:\n        return base\n    left=normalize_timestamp(base).sort_values([\"symbol\",\"decision_timestamp\"])\n    right=normalize_timestamp(layer).sort_values([\"symbol\",\"decision_timestamp\"])\n    cols=[\"symbol\",\"decision_timestamp\"]+[c for c in value_columns if c in right.columns]\n    right=right[cols].drop_duplicates([\"symbol\",\"decision_timestamp\"],keep=\"last\")\n    pieces=[]\n    for symbol,lp in left.groupby(\"symbol\",sort=False):\n        rp=right[right[\"symbol\"].eq(symbol)]\n        if rp.empty:\n            pieces.append(lp); continue\n        m=pd.merge_asof(\n            lp.sort_values(\"decision_timestamp\"),\n            rp.drop(columns=[\"symbol\"]).sort_values(\"decision_timestamp\"),\n            on=\"decision_timestamp\",direction=\"backward\",allow_exact_matches=True,\n            suffixes=(\"\",f\"_{source_name}\")\n        )\n        pieces.append(m)\n    out=pd.concat(pieces,ignore_index=True)\n    for c in (\"fundamental_available_at\",\"fno_observed_at\",\"available_at\"):\n        if c in out.columns:\n            ts=pd.to_datetime(out[c],errors=\"coerce\",utc=True)\n            bad=ts.notna()&ts.gt(out[\"decision_timestamp\"])\n            if bool(bad.any()):\n                raise RuntimeError(f\"{source_name}: {int(bad.sum())} future PIT observations\")\n    return out\n\ndef load_optional_layer(path: Path) -> pd.DataFrame:\n    if not path.exists() or path.stat().st_size==0:\n        return pd.DataFrame()\n    x=pd.read_parquet(path)\n    return normalize_timestamp(x) if not x.empty else pd.DataFrame()\n","colab_build_all_nse_delivery_multifactor_candidate.py":"#!/usr/bin/env python3\n\"\"\"Build ALL-NSE 59-feature multi-factor Delivery candidates in Colab.\"\"\"\nfrom __future__ import annotations\nimport argparse, hashlib, json, threading, time, xml.etree.ElementTree as ET\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\nimport requests\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom delivery_training_universe import benchmark_for_symbol, training_roster\nfrom nse_data_layer import fetch_index_history\nfrom multifactor_data_layer_v2 import FEATURES_59, attach_readiness_flags, load_optional_layer, merge_pit_layer, normalize_symbol, readiness_report\n\nDEFAULT_BULK=\"/content/drive/MyDrive/Indian_Market_Research/production_data/nse_official/bulk_eod\"\nDEFAULT_OUT=\"/content/drive/MyDrive/Indian_Market_Research/models/all_nse_delivery_multifactor_candidate_2026\"\nBASE=Path(\"/content/drive/MyDrive/Indian_Market_Research\")\n\nFUND_PARSER_VERSION=\"2026-10-02-period-context-v3\"\nFUND_CATALOG_CACHE=BASE/\"corporate/nse_integrated_filing_catalog_2025_2026.json\"\nFUND_BROAD=BASE/\"corporate/pit_fundamentals_all_nse_2025_2026.parquet\"\nFUND_BROAD_META=BASE/\"corporate/pit_fundamentals_all_nse_2025_2026.meta.json\"\nFUND_FILING_CACHE=BASE/\"corporate/pit_fundamentals_filing_cache_2025_2026_v3.parquet\"\nFUND_CACHE=BASE/\"corporate/xbrl_cache/all_nse_2025_2026\"\nCANDIDATE_META_NAME=\"CANDIDATE_BUILD_META_2026.json\"\nNSE_HOME=\"https://www.nseindia.com/\"\nNSE_IF_API=\"https://www.nseindia.com/api/integrated-filing-results\"\nFUND_FIELDS={\n \"fundamental_revenue\":(\"RevenueFromOperations\",\"RevenueFromOperationsNetOfExciseDuty\",\"InterestEarned\",\"GrossPremiumIncome\",\"GrossPremiumsWritten\"),\n \"fundamental_pat\":(\"ProfitLossForPeriod\",\"ProfitLossForThePeriod\",\"ProfitLossAfterTax\",\"ProfitLossAfterTaxesMinorityInterestAndShareOfProfitLossOfAssociates\",\"ProfitLossFromOrdinaryActivitiesAfterTax\",\"ProfitLossAfterTaxAndExtraordinaryItems\",\"ProfitLossAfterTaxBeforeExtraordinaryItems\"),\n \"fundamental_eps\":(\"BasicAndDilutedEPSAfterExtraordinaryItemsNetOfTaxExpenseForThePeriodNotToBeAnnualized\",\"BasicEarningsPerShare\",\"DilutedEarningsPerShare\",\"EarningsPerShareBasic\",\"EarningsPerShareDiluted\"),\n \"fundamental_finance_cost\":(\"FinanceCosts\",\"FinanceCost\"),\n \"fundamental_total_equity\":(\"TotalEquity\",\"Equity\",\"EquityAttributableToOwnersOfParent\",\"OtherEquity\",\"Capital\",\"ReservesAndSurplus\"),\n \"fundamental_total_debt\":(\"TotalDebt\",\"Debt\",\"Borrowings\"),\n \"fundamental_cash\":(\"CashAndCashEquivalents\",\"CashAndBankBalances\",\"CashAndBank\"),\n}\n_fund_local=threading.local()\ndef _fund_session():\n    s=getattr(_fund_local,\"s\",None)\n    if s is None:\n        s=requests.Session(); s.headers.update({\"User-Agent\":\"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/134 Safari/537.36\",\"Accept\":\"application/json,text/plain,*/*\",\"Accept-Language\":\"en-GB,en-US;q=0.9,en;q=0.8\",\"Referer\":\"https://www.nseindia.com/companies-listing/corporate-integrated-filing\",\"X-Requested-With\":\"XMLHttpRequest\"})\n        try: s.get(NSE_HOME,timeout=15)\n        except Exception: pass\n        _fund_local.s=s\n    return s\ndef _fd(x):\n    if x is None:\n        return pd.NaT\n    if isinstance(x, pd.Timestamp):\n        return x.tz_localize(\"UTC\") if x.tzinfo is None else x.tz_convert(\"UTC\")\n    s=str(x).strip()\n    if not s:\n        return pd.NaT\n    for fmt in (\"%Y-%m-%d %H:%M:%S\",\"%d-%m-%Y %H:%M:%S\",\"%Y-%m-%d\",\"%d-%m-%Y\"):\n        try:\n            return pd.Timestamp(pd.to_datetime(s,format=fmt,utc=True))\n        except Exception:\n            pass\n    try:\n        return pd.to_datetime(s,errors=\"coerce\",utc=True,dayfirst=True)\n    except Exception:\n        return pd.NaT\ndef _fn(x):\n    try:\n        t=str(x).strip().replace(\",\",\"\")\n        if t.startswith(\"(\") and t.endswith(\")\"): t=\"-\"+t[1:-1]\n        return float(t)\n    except Exception: return np.nan\ndef _fl(tag): return str(tag).split(\"}\",1)[-1].split(\":\",1)[-1]\ndef _catalog_all(force_refresh=False):\n    from datetime import datetime, timezone\n    if not force_refresh and FUND_CATALOG_CACHE.exists() and FUND_CATALOG_CACHE.stat().st_size>0:\n        try:\n            cached=json.loads(FUND_CATALOG_CACHE.read_text(encoding=\"utf-8\"))\n            meta=cached.get(\"meta\",{})\n            same_window=meta.get(\"from_date\")==\"01-01-2025\" and meta.get(\"to_date\")==\"30-09-2026\"\n            fetched=datetime.fromisoformat(meta[\"fetched_at\"]).replace(tzinfo=timezone.utc) if meta.get(\"fetched_at\") else None\n            fresh=fetched is not None and (datetime.now(timezone.utc)-fetched).total_seconds() < 24*3600\n            if same_window and fresh and isinstance(cached.get(\"data\"),list):\n                print(f\"fundamental catalog cache reused rows={len(cached['data']):,}\", flush=True)\n                return cached[\"data\"]\n        except Exception:\n            pass\n\n    s=_fund_session(); out=[]; size=500\n    for page in range(1,401):\n        params={\"type\":\"Integrated Filing- Financials\",\"index\":\"equities\",\"from_date\":\"01-01-2025\",\"to_date\":\"30-09-2026\",\"page\":page,\"size\":size}\n        last=None\n        for attempt in range(3):\n            try:\n                rr=s.get(NSE_IF_API,params=params,timeout=30)\n                rr.raise_for_status()\n                payload=rr.json()\n                batch=payload.get(\"data\",[]) if isinstance(payload,dict) else payload\n                last=None\n                break\n            except Exception as exc:\n                last=exc\n                time.sleep(1.0*(attempt+1))\n        if last:\n            raise last\n        if not batch:\n            break\n        out.extend(batch)\n        if page%20==0:\n            print(f\"fundamental catalog pages={page} rows={len(out):,}\", flush=True)\n        if len(batch)<size:\n            break\n\n    FUND_CATALOG_CACHE.parent.mkdir(parents=True,exist_ok=True)\n    from datetime import datetime, timezone\n    FUND_CATALOG_CACHE.write_text(json.dumps({\n        \"meta\":{\n            \"cache_version\":\"1\",\n            \"from_date\":\"01-01-2025\",\n            \"to_date\":\"30-09-2026\",\n            \"fetched_at\":datetime.now(timezone.utc).isoformat(),\n            \"rows\":len(out),\n        },\n        \"data\":out,\n    },default=str),encoding=\"utf-8\")\n    print(f\"fundamental catalog rows={len(out):,}\", flush=True)\n    return out\n\ndef _fund_filings(catalog,symbols,max_each=4):\n    rows=[]\n    for r in catalog:\n        if str(r.get(\"type\",\"\")).strip().lower()!=\"integrated filing- financials\": continue\n        sym=normalize_symbol(r.get(\"symbol\",\"\")); url=str(r.get(\"xbrl\") or \"\").strip(); pe=_fd(r.get(\"qe_Date\")); av=_fd(r.get(\"broadcast_Date\")); con=str(r.get(\"consolidated\",\"\")).strip().lower()==\"consolidated\"\n        if sym in symbols and url and pd.notna(pe) and pd.notna(av): rows.append({\"symbol\":sym,\"period_end\":pe.normalize(),\"fundamental_available_at\":av,\"xbrl_url\":url,\"consolidated\":con})\n    if not rows: return []\n    d=pd.DataFrame(rows); chosen=[]\n    for (sym,pe),g in d.groupby([\"symbol\",\"period_end\"],sort=False):\n        g=g.sort_values([\"consolidated\",\"fundamental_available_at\"],ascending=[False,False]); chosen.append(g.iloc[0].to_dict())\n    d=pd.DataFrame(chosen).sort_values([\"symbol\",\"period_end\",\"fundamental_available_at\"])\n    return d.groupby(\"symbol\",sort=False,group_keys=False).tail(max_each).to_dict(\"records\")\ndef _fund_cache(url):\n    import hashlib\n    return FUND_CACHE/(hashlib.sha256(url.encode()).hexdigest()[:24]+\".xbrl\")\ndef _fund_xml(f):\n    cp=_fund_cache(f[\"xbrl_url\"]); cp.parent.mkdir(parents=True,exist_ok=True)\n    if cp.exists() and cp.stat().st_size: return cp.read_bytes()\n    s=_fund_session()\n    for attempt in range(3):\n        try:\n            rr=s.get(f[\"xbrl_url\"],headers={\"Accept\":\"*/*\",\"Referer\":NSE_HOME},timeout=30); rr.raise_for_status(); cp.write_bytes(rr.content); return rr.content\n        except Exception:\n            time.sleep(1.0*(attempt+1))\n    return b\"\"\ndef _fund_parse(f):\n    \"\"\"Parse the primary dimension-free PIT financial contexts from an NSE XBRL filing.\n\n    Older versions assumed context IDs named OneD/OneI. NSE filings are not\n    contractually required to use those IDs, so select duration/instant\n    contexts by their periods and the filing period end instead.\n    \"\"\"\n    try:\n        raw = _fund_xml(f)\n        if not raw:\n            return None\n        root=ET.fromstring(raw)\n        xbrli=\"http://www.xbrl.org/2003/instance\"\n        meta={}\n        contexts=[]\n        target_end=pd.Timestamp(f[\"period_end\"]).tz_convert(\"UTC\") if pd.Timestamp(f[\"period_end\"]).tzinfo else pd.Timestamp(f[\"period_end\"],tz=\"UTC\")\n\n        for ctx in root.findall(f\"{{{xbrli}}}context\"):\n            cid=ctx.get(\"id\")\n            if not cid:\n                continue\n            per=ctx.find(f\"{{{xbrli}}}period\")\n            start_ts=end_ts=inst_ts=pd.NaT\n            if per is not None:\n                for e in per:\n                    n=_fl(e.tag).lower()\n                    if n==\"startdate\":\n                        start_ts=_fd(e.text)\n                    elif n==\"enddate\":\n                        end_ts=_fd(e.text)\n                    elif n==\"instant\":\n                        inst_ts=_fd(e.text)\n            dim=any(_fl(e.tag) in (\"explicitmember\",\"typedmember\") for e in ctx.iter())\n            duration_days=np.nan\n            if pd.notna(start_ts) and pd.notna(end_ts):\n                duration_days=(end_ts-start_ts).total_seconds()/86400.0\n            meta[cid]={\"start\":start_ts,\"end\":end_ts,\"instant\":inst_ts,\"dim\":dim,\"duration_days\":duration_days}\n            contexts.append(cid)\n\n        facts={}\n        for e in root.iter():\n            cid=e.get(\"contextRef\")\n            m=meta.get(cid)\n            if not cid or e.text is None or m is None or m[\"dim\"]:\n                continue\n            v=_fn(e.text)\n            if np.isfinite(v):\n                facts.setdefault(cid,{})[_fl(e.tag)]=v\n\n        duration_ids=[]\n        instant_ids=[]\n        for cid,m in meta.items():\n            end=m[\"end\"]\n            if pd.notna(end) and pd.notna(target_end):\n                diff_days=abs((end-target_end).total_seconds()/86400.0)\n            else:\n                diff_days=1e9\n            if pd.notna(m[\"start\"]) and pd.notna(m[\"end\"]) and m[\"duration_days\"] > 0:\n                duration_ids.append((diff_days,m[\"duration_days\"],cid))\n            elif pd.notna(m[\"instant\"]):\n                inst_diff=abs((m[\"instant\"]-target_end).total_seconds()/86400.0) if pd.notna(target_end) else 1e9\n                instant_ids.append((inst_diff,cid))\n\n        # For income/flow fields prefer the shortest clean duration ending on the\n        # filing period (normally the quarter) with non-empty facts. If there is\n        # no exact-period context, use the closest period-end duration.\n        duration_ids.sort(key=lambda x:(x[0],x[1]))\n        q={}\n        for diff_days,duration,cid in duration_ids:\n            if facts.get(cid):\n                q=facts[cid]\n                break\n\n        bs={}\n        for diff_days,cid in sorted(instant_ids):\n            if facts.get(cid):\n                bs=facts[cid]\n                break\n\n        def pick(d,names):\n            for n in names:\n                if n in d and np.isfinite(d[n]):\n                    return float(d[n])\n            return np.nan\n\n        rev=pick(q,FUND_FIELDS[\"fundamental_revenue\"])\n        pat=pick(q,FUND_FIELDS[\"fundamental_pat\"])\n        eps=pick(q,FUND_FIELDS[\"fundamental_eps\"])\n        fin=pick(q,FUND_FIELDS[\"fundamental_finance_cost\"])\n\n        eq=pick(bs,(\"TotalEquity\",\"Equity\",\"EquityAttributableToOwnersOfParent\",\"OtherEquity\"))\n        if not np.isfinite(eq):\n            cap=pick(bs,(\"Capital\",\"EquityShareCapital\"))\n            res=pick(bs,(\"ReservesAndSurplus\",\"OtherEquity\"))\n            eq=(cap+res) if np.isfinite(cap) and np.isfinite(res) else np.nan\n\n        cash=pick(bs,FUND_FIELDS[\"fundamental_cash\"])\n        debt=pick(bs,(\"TotalDebt\",\"Debt\",\"Borrowings\"))\n        cur=pick(bs,(\"BorrowingsCurrent\",))\n        non=pick(bs,(\"BorrowingsNoncurrent\",))\n        if np.isfinite(cur) or np.isfinite(non):\n            debt=sum(v for v in (cur,non) if np.isfinite(v))\n\n        pbt=pick(q,(\"ProfitBeforeTax\",\"ProfitLossFromOrdinaryActivitiesBeforeTax\"))\n        dep=pick(q,(\"DepreciationDepletionAndAmortisationExpense\",\"DepreciationAndAmortisationExpense\",\"Depreciation\"))\n        ebitda=(pbt+(fin if np.isfinite(fin) else 0)+(dep if np.isfinite(dep) else 0)) if np.isfinite(pbt) else np.nan\n\n        vals={\n            \"fundamental_revenue\":rev,\n            \"fundamental_pat\":pat,\n            \"fundamental_eps\":eps,\n            \"fundamental_ebitda\":ebitda,\n            \"fundamental_finance_cost\":fin,\n            \"fundamental_total_equity\":eq,\n            \"fundamental_total_debt\":debt,\n            \"fundamental_cash\":cash,\n        }\n        if not any(np.isfinite(v) for v in vals.values()):\n            return None\n        return {**f,**vals,\"fundamental_source\":\"NSE_XBRL_PIT\",\"fundamental_basis\":\"Integrated_Filing_period_aware_OneD_OneI_fallback\"}\n    except Exception as exc:\n        print(f\"fundamental parse failed for {f.get('symbol', '')}: {type(exc).__name__}: {exc}\", flush=True)\n        return None\n\ndef _funding_key(f):\n    raw=\"|\".join([\n        str(f.get(\"symbol\",\"\")),\n        str(pd.Timestamp(f.get(\"period_end\")).date()) if pd.notna(f.get(\"period_end\")) else \"\",\n        str(pd.Timestamp(f.get(\"fundamental_available_at\")).isoformat()) if pd.notna(f.get(\"fundamental_available_at\")) else \"\",\n        str(f.get(\"xbrl_url\",\"\")),\n    ])\n    return hashlib.sha256(raw.encode(\"utf-8\")).hexdigest()\n\ndef _hash_strings(values):\n    h=hashlib.sha256()\n    for v in sorted(str(x) for x in values):\n        h.update(v.encode(\"utf-8\"))\n        h.update(b\"\\n\")\n    return h.hexdigest()\n\ndef _fund_target_hash(symbols):\n    return _hash_strings(symbols)\n\ndef _load_filing_cache():\n    if not FUND_FILING_CACHE.exists() or FUND_FILING_CACHE.stat().st_size==0:\n        return pd.DataFrame()\n    try:\n        x=pd.read_parquet(FUND_FILING_CACHE)\n        if \"parser_version\" not in x.columns:\n            return pd.DataFrame()\n        x=x[x[\"parser_version\"].eq(FUND_PARSER_VERSION)].copy()\n        return x\n    except Exception:\n        return pd.DataFrame()\n\ndef _save_filing_cache(cache):\n    FUND_FILING_CACHE.parent.mkdir(parents=True,exist_ok=True)\n    cache.to_parquet(FUND_FILING_CACHE,index=False)\n\ndef _parse_one_cached(f):\n    key=_funding_key(f)\n    r=_fund_parse(f)\n    if r is None:\n        return {**f,\"filing_key\":key,\"parser_version\":FUND_PARSER_VERSION,\"parse_status\":\"failed\"}\n    return {**r,\"filing_key\":key,\"parser_version\":FUND_PARSER_VERSION,\"parse_status\":\"parsed\"}\n\ndef _fundamental_build_signature(symbols,filings):\n    return {\n        \"parser_version\":FUND_PARSER_VERSION,\n        \"target_symbols_hash\":_fund_target_hash(symbols),\n        \"target_symbol_count\":len(symbols),\n        \"selected_filing_keys_hash\":_hash_strings([_funding_key(f) for f in filings]),\n        \"selected_filing_count\":len(filings),\n    }\n\ndef _broad_meta_matches(meta,sig):\n    return all(meta.get(k)==v for k,v in sig.items())\n\ndef _bootstrap_cache_from_broad(filing_cache):\n    if not FUND_BROAD.exists() or FUND_BROAD.stat().st_size==0:\n        return filing_cache\n    try:\n        old=pd.read_parquet(FUND_BROAD)\n        if old.empty or \"symbol\" not in old.columns:\n            return filing_cache\n        basis=old.get(\"fundamental_basis\")\n        if basis is None or not basis.astype(str).str.contains(\"period_aware\",case=False,na=False).any():\n            return filing_cache\n        if \"xbrl_url\" not in old.columns or \"period_end\" not in old.columns or \"fundamental_available_at\" not in old.columns:\n            return filing_cache\n        rows=[]\n        for r in old.to_dict(\"records\"):\n            f={k:r.get(k) for k in (\"symbol\",\"period_end\",\"fundamental_available_at\",\"xbrl_url\",\"consolidated\")}\n            rows.append({**r,\"filing_key\":_funding_key(f),\"parser_version\":FUND_PARSER_VERSION,\"parse_status\":\"parsed\"})\n        boot=pd.DataFrame(rows)\n        if filing_cache.empty:\n            return boot\n        return pd.concat([filing_cache,boot],ignore_index=True).drop_duplicates(\"filing_key\",keep=\"last\")\n    except Exception:\n        return filing_cache\n\ndef _ensure_broad_fundamentals(symbols,force_refresh=False,retry_failed=False):\n    catalog=_catalog_all(force_refresh=force_refresh)\n    filings=_fund_filings(catalog,set(symbols),max_each=4)\n    if not filings:\n        raise RuntimeError(\"ALL-NSE PIT fundamentals: no eligible NSE Integrated Filing records found\")\n\n    sig=_fundamental_build_signature(symbols,filings)\n    if not force_refresh and FUND_BROAD.exists() and FUND_BROAD.stat().st_size>0:\n        try:\n            meta=json.loads(FUND_BROAD_META.read_text(encoding=\"utf-8\")) if FUND_BROAD_META.exists() else {}\n            if meta and _broad_meta_matches(meta,sig):\n                print(f\"fundamentals build reused from manifest; rows={int(meta.get('rows',0)):,}\",flush=True)\n                return {\"path\":str(FUND_BROAD),\"rebuilt\":False,\"cached\":True,\"symbols\":int(meta.get(\"symbols\",0)),\"rows\":int(meta.get(\"rows\",0))}\n            # One-time bootstrap for a broad file produced by the period-aware\n            # parser before the persistent build metadata existed. This prevents\n            # an immediate second full NSE/XBRL pass after a completed run.\n            if not meta:\n                old=pd.read_parquet(FUND_BROAD)\n                basis=old.get(\"fundamental_basis\")\n                compatible = basis is not None and basis.astype(str).str.contains(\"period_aware\",case=False,na=False).any()\n                if compatible and \"symbol\" in old.columns:\n                    boot={\n                        **sig,\n                        \"rows\":int(len(old)),\n                        \"symbols\":int(old[\"symbol\"].map(normalize_symbol).nunique()),\n                        \"bootstrapped_from_existing_broad\":True,\n                    }\n                    FUND_BROAD_META.write_text(json.dumps(boot,indent=2,default=str),encoding=\"utf-8\")\n                    print(\n                        f\"fundamentals build metadata bootstrapped from existing broad file; \"\n                        f\"rows={len(old):,} symbols={old['symbol'].map(normalize_symbol).nunique():,}\",\n                        flush=True,\n                    )\n                    return {\"path\":str(FUND_BROAD),\"rebuilt\":False,\"cached\":True,\n                            \"symbols\":int(boot[\"symbols\"]),\"rows\":int(boot[\"rows\"])}\n        except Exception:\n            pass\n\n    filing_cache=_bootstrap_cache_from_broad(_load_filing_cache())\n    cached_keys=set(filing_cache[\"filing_key\"].astype(str)) if not filing_cache.empty else set()\n    selected_by_key={_funding_key(f):f for f in filings}\n    missing=[f for k,f in selected_by_key.items() if k not in cached_keys]\n    failed_cached=set()\n    if not filing_cache.empty and \"parse_status\" in filing_cache.columns:\n        failed_cached=set(filing_cache.loc[filing_cache[\"parse_status\"].eq(\"failed\"),\"filing_key\"].astype(str))\n    if retry_failed:\n        missing.extend(selected_by_key[k] for k in failed_cached if k in selected_by_key and k not in {_funding_key(x) for x in missing})\n\n    print(\n        f\"fundamentals cache: selected={len(filings):,} \"\n        f\"cached={len(filings)-len(missing):,} \"\n        f\"to_process={len(missing):,} \"\n        f\"failed_cached={len(failed_cached):,}\",\n        flush=True,\n    )\n\n    new_rows=[]\n    if missing:\n        with ThreadPoolExecutor(max_workers=16) as ex:\n            futs=[ex.submit(_parse_one_cached,f) for f in missing]\n            for i,fut in enumerate(as_completed(futs),1):\n                new_rows.append(fut.result())\n                if i%500==0:\n                    parsed_now=sum(r.get(\"parse_status\")==\"parsed\" for r in new_rows)\n                    print(f\"fundamentals parse={i:,}/{len(missing):,} new_parsed={parsed_now:,}\",flush=True)\n\n    if new_rows:\n        filing_cache=pd.concat([filing_cache,pd.DataFrame(new_rows)],ignore_index=True)\n    if not filing_cache.empty:\n        filing_cache=filing_cache.drop_duplicates(\"filing_key\",keep=\"last\")\n        _save_filing_cache(filing_cache)\n\n    selected_keys=set(selected_by_key)\n    usable=filing_cache[\n        filing_cache[\"filing_key\"].astype(str).isin(selected_keys) &\n        filing_cache[\"parse_status\"].eq(\"parsed\")\n    ].copy()\n    if usable.empty:\n        raise RuntimeError(\"ALL-NSE PIT fundamentals: selected filings have no parsed records\")\n\n    d=pd.DataFrame(usable).drop(columns=[\"filing_key\",\"parser_version\",\"parse_status\"],errors=\"ignore\")\n    d[\"symbol\"]=d[\"symbol\"].map(normalize_symbol)\n    d[\"period_end\"]=pd.to_datetime(d[\"period_end\"],utc=True).dt.normalize()\n    d[\"fundamental_available_at\"]=pd.to_datetime(d[\"fundamental_available_at\"],utc=True)\n    for c in FUND_FIELDS:\n        if c in d.columns:\n            d[c]=pd.to_numeric(d[c],errors=\"coerce\")\n    d=d.sort_values([\"symbol\",\"period_end\",\"fundamental_available_at\"]).drop_duplicates([\"symbol\",\"period_end\"],keep=\"last\")\n    pieces=[]\n    for sym,g in d.groupby(\"symbol\",sort=False):\n        g=g.sort_values(\"period_end\").copy()\n        for base,growth in ((\"fundamental_revenue\",\"fund_revenue_growth\"),(\"fundamental_pat\",\"fund_pat_growth\"),(\"fundamental_eps\",\"fund_eps_growth\"),(\"fundamental_ebitda\",\"fund_ebitda_growth\")):\n            g[growth]=g[base].pct_change(fill_method=None)\n        g[\"fund_pat_margin\"]=g[\"fundamental_pat\"]/g[\"fundamental_revenue\"].replace(0,np.nan)\n        g[\"fund_debt_to_equity\"]=g[\"fundamental_total_debt\"]/g[\"fundamental_total_equity\"].replace(0,np.nan)\n        g[\"fund_cash_to_debt\"]=g[\"fundamental_cash\"]/g[\"fundamental_total_debt\"].replace(0,np.nan)\n        pieces.append(g)\n    d=pd.concat(pieces,ignore_index=True).replace([np.inf,-np.inf],np.nan)\n    FUND_BROAD.parent.mkdir(parents=True,exist_ok=True)\n    d.to_parquet(FUND_BROAD,index=False)\n    report={\n        \"path\":str(FUND_BROAD),\n        \"rows\":len(d),\n        \"symbols\":int(d[\"symbol\"].nunique()),\n        \"target_symbols\":len(symbols),\n        \"symbol_coverage_pct\":round(100*d[\"symbol\"].nunique()/len(symbols),3) if symbols else 0.0,\n        \"filings_selected\":len(filings),\n        \"filings_parsed\":int((filing_cache[\"filing_key\"].astype(str).isin(selected_keys) & filing_cache[\"parse_status\"].eq(\"parsed\")).sum()),\n        \"filings_failed_cached\":int((filing_cache[\"filing_key\"].astype(str).isin(selected_keys) & filing_cache[\"parse_status\"].eq(\"failed\")).sum()),\n        \"feature_coverage_pct\":{c:round(100*d[c].notna().mean(),3) for c in FUND_FIELDS if c in d.columns},\n        **sig,\n    }\n    FUND_BROAD_META.write_text(json.dumps(report,indent=2,default=str),encoding=\"utf-8\")\n    print(json.dumps(report,indent=2,default=str),flush=True)\n    return report\n\nif len(FEATURES_59) != 59:\n    raise RuntimeError(f\"Feature contract must contain exactly 59 features; found {len(FEATURES_59)}\")\n\ndef sha256(path: Path)->str:\n    h=hashlib.sha256()\n    with path.open(\"rb\") as f:\n        for b in iter(lambda:f.read(1024*1024),b\"\"): h.update(b)\n    return h.hexdigest()\n\ndef load_bulk(root: Path)->pd.DataFrame:\n    files=sorted(root.rglob(\"*.parquet\"))+sorted(root.rglob(\"*.csv\"))\n    if not files: raise FileNotFoundError(f\"No EOD bulk files under {root}\")\n    parts=[]\n    for p in files:\n        try: x=pd.read_parquet(p) if p.suffix.lower()==\".parquet\" else pd.read_csv(p)\n        except Exception: continue\n        if x.empty: continue\n        ren={}\n        for c in x.columns:\n            lc=str(c).strip().lower()\n            mp={\"ticker\":\"symbol\",\"tckrsymb\":\"symbol\",\"trade_date\":\"date\",\"trad_dt\":\"date\",\n                \"open_price\":\"open\",\"opnpric\":\"open\",\"high_price\":\"high\",\"hghpric\":\"high\",\n                \"low_price\":\"low\",\"lwpric\":\"low\",\"close_price\":\"close\",\"clspric\":\"close\",\n                \"traded_volume\":\"volume\",\"ttltradgvol\":\"volume\",\"dly_pct\":\"delivery_pct\",\n                \"dly_qty\":\"delivery_qty\"}\n            if lc in (\"symbol\",\"date\",\"open\",\"high\",\"low\",\"close\",\"volume\",\"delivery_pct\",\"delivery_qty\"): ren[c]=lc\n            elif lc in mp: ren[c]=mp[lc]\n        x=x.rename(columns=ren)\n        req={\"symbol\",\"date\",\"open\",\"high\",\"low\",\"close\",\"volume\"}\n        if not req.issubset(x.columns): continue\n        x[\"symbol\"]=x[\"symbol\"].map(normalize_symbol)\n        x[\"date\"]=pd.to_datetime(x[\"date\"],errors=\"coerce\").dt.normalize()\n        for c in (\"open\",\"high\",\"low\",\"close\",\"volume\",\"delivery_pct\",\"delivery_qty\"):\n            if c in x.columns: x[c]=pd.to_numeric(x[c],errors=\"coerce\")\n        keep=list(req|set([\"delivery_pct\",\"delivery_qty\"])&set(x.columns))\n        parts.append(x[keep])\n    if not parts: raise RuntimeError(\"No readable EOD bulk dataset\")\n    return pd.concat(parts,ignore_index=True).dropna(subset=[\"symbol\",\"date\",\"open\",\"high\",\"low\",\"close\"]).sort_values([\"symbol\",\"date\"]).drop_duplicates([\"symbol\",\"date\"],keep=\"last\").reset_index(drop=True)\n\ndef rsi(s,n=14):\n    d=s.diff(); g=d.clip(lower=0).rolling(n,min_periods=n).mean(); l=(-d.clip(upper=0)).rolling(n,min_periods=n).mean()\n    return (100-100/(1+g/l.replace(0,np.nan))).where(l.ne(0),100.0)\n\ndef add_features(x):\n    z=x.sort_values(\"date\").copy(); c=z[\"close\"]; v=z[\"volume\"]\n    for n in (1,3,6,12,24,48): z[f\"ret_{n}\"]=c.pct_change(n)\n    z[\"rsi14\"]=rsi(c,14)\n    prev=c.shift(1); tr=pd.concat([(z[\"high\"]-z[\"low\"]),(z[\"high\"]-prev).abs(),(z[\"low\"]-prev).abs()],axis=1).max(axis=1)\n    z[\"atr14\"]=tr.ewm(alpha=1/14,adjust=False,min_periods=14).mean()\n    ma20=c.rolling(20).mean(); ma50=c.rolling(50).mean(); vm=v.rolling(20).mean(); vs=v.rolling(20).std()\n    z[\"ma20_ratio\"]=c/ma20-1; z[\"ma50_ratio\"]=c/ma50-1; z[\"volume_ratio\"]=v/vm; z[\"volume_z20\"]=(v-vm)/vs\n    z[\"range_pct\"]=(z[\"high\"]-z[\"low\"])/c; z[\"body_pct\"]=(c-z[\"open\"])/z[\"open\"]; z[\"up_bar\"]=(c>z[\"open\"]).astype(float)\n    for c0 in (\"delivery_pct\",\"delivery_qty\"):\n        if c0 not in z.columns: z[c0]=np.nan\n    dp=z[\"delivery_pct\"]; dq=z[\"delivery_qty\"]\n    z[\"delivery_pct_change_1\"]=dp.pct_change(fill_method=None); z[\"delivery_pct_ratio20\"]=dp/dp.rolling(20).mean()\n    z[\"delivery_pct_z20\"]=(dp-dp.rolling(20).mean())/dp.rolling(20).std()\n    z[\"delivery_qty_ratio20\"]=dq/dq.rolling(20).mean(); z[\"delivery_qty_z20\"]=(dq-dq.rolling(20).mean())/dq.rolling(20).std()\n    return z\n\ndef _official_market_context(index_symbol: str, start_date, end_date) -> pd.DataFrame:\n    \"\"\"Load official NSE benchmark history from Drive first, API second.\"\"\"\n    frame = fetch_index_history(\n        index_symbol,\n        pd.Timestamp(start_date).date(),\n        pd.Timestamp(end_date).date(),\n        sync_drive=True,\n    )\n    if frame is None or frame.empty:\n        raise RuntimeError(\n            f\"Official NSE benchmark history unavailable for {index_symbol}: \"\n            f\"{start_date}..{end_date}\"\n        )\n\n    x = frame.copy()\n    x[\"date\"] = pd.to_datetime(x[\"date\"], errors=\"coerce\").dt.normalize()\n    x[\"close\"] = pd.to_numeric(x[\"close\"], errors=\"coerce\")\n    x = (\n        x.dropna(subset=[\"date\", \"close\"])\n         .sort_values(\"date\")\n         .drop_duplicates(\"date\", keep=\"last\")\n         .reset_index(drop=True)\n    )\n\n    c0 = x[\"close\"]\n    d = c0.diff()\n    gain = d.clip(lower=0).rolling(14, min_periods=14).mean()\n    loss = (-d.clip(upper=0)).rolling(14, min_periods=14).mean()\n    rsi14 = (\n        100.0 - 100.0 / (1.0 + gain / loss.replace(0, np.nan))\n    ).where(loss.ne(0), 100.0)\n\n    out = pd.DataFrame({\"date\": x[\"date\"]})\n    for n in (1, 3, 6, 12):\n        out[f\"market_ret_{n}\"] = c0.pct_change(n)\n    out[\"market_rsi14\"] = rsi14\n    out[\"market_mood_proxy\"] = out[\"market_ret_1\"]\n    out[\"market_stress_proxy\"] = c0.pct_change().rolling(20, min_periods=20).std()\n    return out\n\n\ndef market_context_tables(bulk) -> dict[str, pd.DataFrame]:\n    start_date = pd.Timestamp(bulk[\"date\"].min()).normalize()\n    end_date = pd.Timestamp(bulk[\"date\"].max()).normalize()\n    return {\n        \"^NSEI\": _official_market_context(\"^NSEI\", start_date, end_date),\n        \"^NSEBANK\": _official_market_context(\"^NSEBANK\", start_date, end_date),\n    }\n\n\ndef make_panel(bulk, symbols):\n    # Normalize the bulk symbol column before filtering. The training roster\n    # is built on normalized symbols; filtering against raw symbol text can\n    # silently collapse the panel when archive symbols carry series/suffix\n    # variants such as '-EQ'.\n    work = bulk.copy()\n    work[\"_symbol_norm\"] = work[\"symbol\"].map(normalize_symbol)\n    ctxs = market_context_tables(work)\n    parts = []\n    for sym in symbols:\n        z = add_features(work[work[\"_symbol_norm\"].eq(sym)].drop(columns=[\"_symbol_norm\"]).copy())\n        if z.empty:\n            continue\n        benchmark = benchmark_for_symbol(sym)\n        ctx = ctxs.get(benchmark)\n        if ctx is None or ctx.empty:\n            raise RuntimeError(f\"Missing official NSE benchmark context for {sym}: {benchmark}\")\n        z = z.merge(ctx, on=\"date\", how=\"left\")\n        for n in (1, 6, 12):\n            z[f\"relative_ret_{n}\"] = z[f\"ret_{n}\"] - z[f\"market_ret_{n}\"]\n        z[\"decision_timestamp\"] = z[\"date\"] + pd.Timedelta(hours=15, minutes=30)\n        parts.append(z)\n    return pd.concat(parts, ignore_index=True).sort_values([\"symbol\", \"decision_timestamp\"]).reset_index(drop=True)\n\ndef merge_optional(panel):\n    out=panel.copy(); audit={}\n    sources=[\n      (\"news\",BASE/\"processed/pit_replay/NEWS_ENRICHED_PIT_REPLAY_2026.parquet\",[\"news_available\"]),\n      (\"events\",BASE/\"processed/pit_replay/EVENTS_FUNDAMENTALS_PIT_REPLAY_2026.parquet\",[\"event_available\",\"event_high_impact\"]),\n      (\"fno\",BASE/\"processed/pit_replay/FNO_PIT_REPLAY_2026.parquet\",[\"fno_pcr\",\"fno_oi_change\",\"fno_iv_change\",\"fno_available\",\"fno_fut_close\",\"fno_fut_return\",\"fno_fut_oi\"]),\n    ]\n    for kind,path,cols in sources:\n        layer=load_optional_layer(path)\n        if layer.empty: audit[kind]={\"exists\":False,\"rows\":0}; continue\n        out=merge_pit_layer(out,layer,cols,kind)\n        audit[kind]={\"exists\":True,\"rows\":int(len(layer)),\"symbols\":int(layer[\"symbol\"].nunique())}\n    # Derived F&O basis is PIT-safe because fno_fut_close is already merged\n    # with the same backward-as-of F&O timestamp contract.\n    if \"fno_fut_basis\" not in out.columns:\n        if \"fno_fut_close\" in out.columns:\n            fut=pd.to_numeric(out[\"fno_fut_close\"],errors=\"coerce\")\n            spot=pd.to_numeric(out[\"close\"],errors=\"coerce\")\n            out[\"fno_fut_basis\"]=(fut/spot.replace(0,np.nan))-1.0\n        else:\n            out[\"fno_fut_basis\"]=np.nan\n\n    # ONLY the broad ALL-NSE PIT fundamentals layer is accepted here.\n    # Never silently fall back to the legacy three-stock pilot.\n    fp=BASE/\"corporate/pit_fundamentals_all_nse_2025_2026.parquet\"\n    fp=fp if fp.exists() and fp.stat().st_size>0 else None\n    if fp is not None:\n        f=pd.read_parquet(fp)\n        f[\"symbol\"]=f[\"symbol\"].map(normalize_symbol)\n        f[\"fundamental_available_at\"]=pd.to_datetime(\n            f[\"fundamental_available_at\"],errors=\"coerce\",utc=True\n        )\n        f[\"period_end\"]=pd.to_datetime(\n            f.get(\"period_end\"),errors=\"coerce\",utc=True\n        ).dt.normalize() if \"period_end\" in f.columns else pd.NaT\n        f=f.dropna(subset=[\"symbol\",\"fundamental_available_at\"]).sort_values(\n            [\"symbol\",\"fundamental_available_at\"]\n        )\n        vals=[\n            \"fundamental_revenue\",\"fundamental_pat\",\"fundamental_eps\",\n            \"fundamental_ebitda\",\"fundamental_finance_cost\",\n            \"fundamental_total_equity\",\"fundamental_total_debt\",\"fundamental_cash\",\n            \"fund_revenue_growth\",\"fund_pat_growth\",\"fund_eps_growth\",\n            \"fund_ebitda_growth\",\"fund_pat_margin\",\"fund_debt_to_equity\",\n            \"fund_cash_to_debt\",\n        ]\n        pieces=[]\n        for sym,lp in out.groupby(\"symbol\",sort=False):\n            rp=f[f[\"symbol\"].eq(sym)]\n            if rp.empty:\n                pieces.append(lp)\n                continue\n            cols=[\"fundamental_available_at\"]+[x for x in vals if x in rp.columns]\n            rp=rp[cols].drop_duplicates([\"fundamental_available_at\"],keep=\"last\")\n            m=pd.merge_asof(\n                lp.sort_values(\"decision_timestamp\"),\n                rp.sort_values(\"fundamental_available_at\"),\n                left_on=\"decision_timestamp\",\n                right_on=\"fundamental_available_at\",\n                direction=\"backward\",\n                allow_exact_matches=True,\n            )\n            pieces.append(m)\n        if pieces:\n            out=pd.concat(pieces,ignore_index=True)\n        # Fundamental readiness is based on the existence of an as-of filing,\n        # not on every optional accounting field being present.\n        fa=pd.to_datetime(out.get(\"fundamental_available_at\"),errors=\"coerce\",utc=True)\n        valid=fa.notna() & fa.le(out[\"decision_timestamp\"])\n        out[\"fundamentals_available\"]=valid.astype(\"Int8\")\n        out[\"fundamental_age_days\"]=np.nan\n        out.loc[valid,\"fundamental_age_days\"]=(\n            out.loc[valid,\"decision_timestamp\"]-fa.loc[valid]\n        ).dt.total_seconds()/86400.0\n        out[\"fundamental_stale_90d\"]=pd.Series(\n            np.where(valid,(out[\"fundamental_age_days\"]>90).astype(float),np.nan),\n            index=out.index,dtype=\"float64\"\n        )\n        out[\"fundamental_stale_180d\"]=pd.Series(\n            np.where(valid,(out[\"fundamental_age_days\"]>180).astype(float),np.nan),\n            index=out.index,dtype=\"float64\"\n        )\n        audit[\"fundamentals\"]={\n            \"exists\":True,\n            \"path\":str(fp),\n            \"rows\":int(len(f)),\n            \"symbols\":int(f[\"symbol\"].nunique()),\n        }\n    else:\n        audit[\"fundamentals\"]={\"exists\":False,\"rows\":0}\n        out[\"fundamentals_available\"]=pd.Series(np.zeros(len(out),dtype=\"Int8\"),index=out.index)\n        out[\"fundamental_age_days\"]=np.nan\n        out[\"fundamental_stale_90d\"]=np.nan\n        out[\"fundamental_stale_180d\"]=np.nan\n    return attach_readiness_flags(out),audit\n\ndef add_targets(panel,steps,name):\n    x=panel.sort_values([\"symbol\",\"decision_timestamp\"]).copy()\n    with np.errstate(divide=\"ignore\",invalid=\"ignore\",over=\"ignore\"):\n        x[f\"target_return_{name}\"]=x.groupby(\"symbol\")[\"close\"].shift(-steps)/x[\"close\"]-1.0\n    x[f\"target_return_{name}\"]=x[f\"target_return_{name}\"].replace([np.inf,-np.inf],np.nan)\n    x[f\"target_up_{name}\"]=(x[f\"target_return_{name}\"]>0).astype(int)\n    return x.dropna(subset=[f\"target_return_{name}\"])\n\ndef fit_models(frame,horizon):\n    cutoff=frame[\"decision_timestamp\"].max()-pd.Timedelta(days=60); tr=frame[frame[\"decision_timestamp\"]<cutoff]; te=frame[frame[\"decision_timestamp\"]>=cutoff]\n    if tr.empty or te.empty or tr[f\"target_up_{horizon}\"].nunique()<2: raise RuntimeError(f\"{horizon}: insufficient chronological train/holdout data\")\n    Xtr=tr[FEATURES_59].apply(pd.to_numeric,errors=\"coerce\").replace([np.inf,-np.inf],np.nan)\n    Xte=te[FEATURES_59].apply(pd.to_numeric,errors=\"coerce\").replace([np.inf,-np.inf],np.nan)\n    yreg=tr[f\"target_return_{horizon}\"].replace([np.inf,-np.inf],np.nan)\n    good=yreg.notna()\n    Xtr=Xtr.loc[good]\n    yreg=yreg.loc[good]\n    yclf=tr.loc[good,f\"target_up_{horizon}\"].astype(int)\n    if Xtr.empty or yclf.nunique()<2:\n        raise RuntimeError(f\"{horizon}: finite training set is insufficient after non-finite-value cleanup\")\n    clf=Pipeline([(\"imputer\",SimpleImputer(strategy=\"median\",add_indicator=True,keep_empty_features=True)),(\"scale\",StandardScaler(with_mean=False)),(\"model\",LogisticRegression(C=.5,class_weight=\"balanced\",solver=\"lbfgs\",max_iter=2500,random_state=42))])\n    reg=Pipeline([(\"imputer\",SimpleImputer(strategy=\"median\",add_indicator=True,keep_empty_features=True)),(\"model\",HistGradientBoostingRegressor(max_iter=180,learning_rate=.04,max_leaf_nodes=19,l2_regularization=2.0,random_state=42))])\n    clf.fit(Xtr,tr[f\"target_up_{horizon}\"].astype(int))\n    reg.fit(Xtr,tr[f\"target_return_{horizon}\"].astype(float))\n    for fitted in (clf,reg):\n        imp=fitted.named_steps.get(\"imputer\")\n        learned=int(getattr(imp,\"n_features_in_\",0)) if imp is not None else 0\n        if not learned and imp is not None and getattr(imp,\"statistics_\",None) is not None:\n            learned=int(len(imp.statistics_))\n        if learned!=len(FEATURES_59):\n            raise RuntimeError(f\"{horizon}: fitted model contract has {learned} original features; expected {len(FEATURES_59)}\")\n    p=clf.predict_proba(Xte)[:,1]; pr=reg.predict(Xte); y=te[f\"target_up_{horizon}\"].to_numpy()\n    return clf,reg,{\"train_rows\":int(len(tr)),\"holdout_rows\":int(len(te)),\"holdout_accuracy\":float(np.mean((p>=.5)==y)),\"mean_abs_return_error\":float(np.mean(np.abs(pr-te[f\"target_return_{horizon}\"].to_numpy()))),\"holdout_mean_readiness_pct\":float(te[\"data_readiness_score\"].mean()*100)}\n\ndef _candidate_input_signature(bulk_root,bulk,trainable,fund_report):\n    optional=[\n        BASE/\"processed/pit_replay/NEWS_ENRICHED_PIT_REPLAY_2026.parquet\",\n        BASE/\"processed/pit_replay/EVENTS_FUNDAMENTALS_PIT_REPLAY_2026.parquet\",\n        BASE/\"processed/pit_replay/FNO_PIT_REPLAY_2026.parquet\",\n        BASE/\"corporate/pit_fundamentals_all_nse_2025_2026.parquet\",\n    ]\n    sig={\n        \"engine_version\":\"2026-all-nse-delivery-multifactor-59-candidate-v2\",\n        \"feature_contract_hash\":hashlib.sha256(json.dumps(FEATURES_59,separators=(\",\",\":\")).encode()).hexdigest(),\n        \"feature_count\":len(FEATURES_59),\n        \"bulk_root\":str(bulk_root),\n        \"bulk_rows\":int(len(bulk)),\n        \"bulk_symbols\":int(bulk[\"symbol\"].nunique()),\n        \"bulk_min_date\":str(pd.Timestamp(bulk[\"date\"].min()).date()),\n        \"bulk_max_date\":str(pd.Timestamp(bulk[\"date\"].max()).date()),\n        \"trainable_hash\":_fund_target_hash(trainable),\n        \"fundamental_meta_hash\":sha256(FUND_BROAD_META) if FUND_BROAD_META.exists() else \"\",\n    }\n    for p in optional[:-1]:\n        sig[str(p)]=({\"exists\":True,\"size\":p.stat().st_size,\"mtime_ns\":p.stat().st_mtime_ns} if p.exists() else {\"exists\":False})\n    return sig\n\ndef _candidate_artifacts_complete(out):\n    mp=out/\"ALL_NSE_MULTIFACTOR_59_MODEL_MANIFEST_2026.json\"\n    if not mp.exists() or mp.stat().st_size==0:\n        return False\n    try:\n        m=json.loads(mp.read_text(encoding=\"utf-8\"))\n    except Exception:\n        return False\n    if m.get(\"promotion_status\")!=\"CANDIDATE_ONLY\" or m.get(\"features\")!=FEATURES_59:\n        return False\n    for item in m.get(\"artifacts\",{}).values():\n        if not isinstance(item,dict) or item.get(\"status\")!=\"CANDIDATE_READY\":\n            return False\n        for k in (\"classifier\",\"regressor\"):\n            if not Path(item[k]).exists():\n                return False\n    return True\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\"--bulk-root\",default=DEFAULT_BULK)\n    ap.add_argument(\"--out\",default=DEFAULT_OUT)\n    ap.add_argument(\"--min-history\",type=int,default=60)\n    ap.add_argument(\"--refresh-fundamentals\",action=\"store_true\")\n    ap.add_argument(\"--retry-failed-fundamentals\",action=\"store_true\")\n    ap.add_argument(\"--force-rebuild-candidate\",action=\"store_true\")\n    args=ap.parse_args()\n\n    out=Path(args.out)\n    out.mkdir(parents=True,exist_ok=True)\n    bulk_root=Path(args.bulk_root)\n    bulk=load_bulk(bulk_root)\n    roster=training_roster(bulk,min_history=args.min_history)\n    trainable=list(roster[\"trainable_symbols\"])\n\n    fund_report=_ensure_broad_fundamentals(\n        trainable,\n        force_refresh=args.refresh_fundamentals,\n        retry_failed=args.retry_failed_fundamentals,\n    )\n\n    universe={\n        \"universe_source\":roster.get(\"universe_source\"),\n        \"universe_as_of\":roster.get(\"universe_as_of\"),\n        \"universe_symbol_count\":int(roster[\"universe_symbol_count\"]),\n        \"symbols_with_bulk_history\":int(roster[\"symbols_with_bulk_history\"]),\n        \"trainable_symbol_count\":int(roster[\"trainable_symbol_count\"]),\n        \"excluded_symbol_count\":int(roster[\"excluded_symbol_count\"]),\n        \"minimum_history_sessions\":int(roster[\"minimum_history_sessions\"]),\n    }\n\n    input_sig=_candidate_input_signature(bulk_root,bulk,trainable,fund_report)\n    candidate_meta=out/CANDIDATE_META_NAME\n    if not args.force_rebuild_candidate and not args.refresh_fundamentals and candidate_meta.exists() and _candidate_artifacts_complete(out):\n        try:\n            old=json.loads(candidate_meta.read_text(encoding=\"utf-8\"))\n            if old.get(\"input_signature\")==input_sig:\n                print(\"CANDIDATE BUILD REUSED; inputs and feature contract unchanged.\",flush=True)\n                return\n        except Exception:\n            pass\n\n    panel,audit=merge_optional(make_panel(bulk,trainable))\n    (out/\"DATA_READINESS_REPORT_2026.json\").write_text(\n        json.dumps({\"universe\":universe,\"readiness\":readiness_report(panel),\"optional_sources\":audit},\n                   indent=2,default=str),\n        encoding=\"utf-8\",\n    )\n    panel.to_parquet(out/\"ALL_NSE_MULTIFACTOR_59_FEATURE_PIT_PANEL_2026.parquet\",index=False)\n\n    artifacts={}\n    for horizon,steps in ((\"next_session\",1),(\"3d\",3),(\"5d\",5),(\"7d\",7)):\n        cp=out/f\"deployment_all_nse_{horizon}_multifactor59_classifier_2026.joblib\"\n        rp=out/f\"deployment_all_nse_{horizon}_multifactor59_regressor_2026.joblib\"\n        try:\n            d=add_targets(panel,steps,horizon)\n            clf,reg,metrics=fit_models(d,horizon)\n            # Save only after the 59-input contract has been checked.\n            joblib.dump(clf,cp)\n            joblib.dump(reg,rp)\n            artifacts[horizon]={\n                \"status\":\"CANDIDATE_READY\",\n                \"classifier\":str(cp),\n                \"regressor\":str(rp),\n                \"classifier_sha256\":sha256(cp),\n                \"regressor_sha256\":sha256(rp),\n                \"metrics\":metrics,\n            }\n        except Exception as exc:\n            if cp.exists():\n                cp.unlink()\n            if rp.exists():\n                rp.unlink()\n            artifacts[horizon]={\n                \"status\":\"BLOCKED\",\n                \"reason\":f\"{type(exc).__name__}: {exc}\",\n            }\n\n    manifest={\n        \"engine_version\":\"2026-all-nse-delivery-multifactor-59-candidate-v2\",\n        \"promotion_status\":\"CANDIDATE_ONLY\",\n        \"universe\":universe,\n        \"feature_count\":len(FEATURES_59),\n        \"features\":FEATURES_59,\n        \"feature_contract_hash\":input_sig[\"feature_contract_hash\"],\n        \"artifacts\":artifacts,\n        \"data_readiness_path\":str(out/\"DATA_READINESS_REPORT_2026.json\"),\n        \"pit_policy\":\"Only observations at or before decision_timestamp are eligible. Missing information is not fabricated.\",\n        \"no_auto_orders\":True,\n    }\n    (out/\"ALL_NSE_MULTIFACTOR_59_MODEL_MANIFEST_2026.json\").write_text(\n        json.dumps(manifest,indent=2,default=str),\n        encoding=\"utf-8\",\n    )\n    candidate_meta.write_text(json.dumps({\n        \"input_signature\":input_sig,\n        \"manifest_path\":str(out/\"ALL_NSE_MULTIFACTOR_59_MODEL_MANIFEST_2026.json\"),\n        \"fundamentals_report\":fund_report,\n    },indent=2,default=str),encoding=\"utf-8\")\n    print(json.dumps(manifest,indent=2,default=str),flush=True)\n\nif __name__==\"__main__\":\n    main()\n","colab_run_multifactor_upgrade.py":"#!/usr/bin/env python3\n\"\"\"One-command Google Colab runner for the multi-factor candidate.\"\"\"\nfrom __future__ import annotations\nimport argparse\nimport os\nimport subprocess\nimport sys\nfrom pathlib import Path\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--project\", default=\"/content/indian-market-trading\")\n    ap.add_argument(\"--drive-root\", default=\"/content/drive/MyDrive/Indian_Market_Research\")\n    ap.add_argument(\"--bulk-root\", default=None, help=\"Canonical NSE bulk EOD archive.\")\n    ap.add_argument(\"--refresh-fundamentals\", action=\"store_true\", help=\"Force a fresh NSE fundamentals catalog/XBRL pass.\")\n    ap.add_argument(\"--retry-failed-fundamentals\", action=\"store_true\", help=\"Retry only filings previously cached as failed.\")\n    ap.add_argument(\"--force-rebuild-candidate\", action=\"store_true\", help=\"Force panel/model regeneration even when inputs are unchanged.\")\n    ap.add_argument(\"--preflight\", action=\"store_true\", help=\"Check code contract and Drive caches without running the build.\")\n    a = ap.parse_args()\n\n    def log(msg):\n        print(msg, flush=True)\n\n    log(\"COLAB MULTIFACTOR UPGRADE STARTING\",)\n    log(f\"Project: {a.project}\")\n    log(f\"Drive root: {a.drive_root}\")\n    if a.preflight:\n        from multifactor_data_layer_v2 import FEATURES_59\n        if len(FEATURES_59) != 59:\n            raise SystemExit(f\"PRECHECK FAILED: feature count={len(FEATURES_59)}\")\n        builder_path = Path(a.project) / \"colab_build_all_nse_delivery_multifactor_candidate.py\"\n        builder_text = builder_path.read_text(encoding=\"utf-8\")\n        required = [\n            \"2026-all-nse-delivery-multifactor-59-candidate-v2\",\n            \"pit_fundamentals_filing_cache_2025_2026_v3\",\n            \"CANDIDATE_BUILD_META_2026.json\",\n        ]\n        missing = [x for x in required if x not in builder_text]\n        if missing:\n            raise SystemExit(\"PRECHECK FAILED: stale builder; missing=\" + \",\".join(missing))\n        log(\"PRECHECK OK: feature contract=59; incremental-cache builder present.\")\n        return 0\n\n    try:\n        from google.colab import drive\n        if not Path(\"/content/drive/MyDrive\").exists():\n            log(\"Mounting Google Drive...\")\n            drive.mount(\"/content/drive\")\n    except Exception as exc:\n        log(f\"Drive mount check: {type(exc).__name__}: {exc}\")\n\n    p = Path(a.project).resolve()\n    d = Path(a.drive_root).resolve()\n    if not p.exists():\n        raise SystemExit(f\"Project not found: {p}\")\n\n    if a.bulk_root:\n        bulk = Path(a.bulk_root).resolve()\n    else:\n        candidates = [\n            d / \"production_data/nse_official/bulk_eod\",\n            d / \"processed/universe_expansion_2026_v4/bulk_eod\",\n        ]\n        bulk = next((x for x in candidates if x.exists()), candidates[0])\n\n    if not bulk.exists():\n        raise SystemExit(\n            \"NSE bulk EOD archive not found. Checked:\\n  \"\n            + \"\\n  \".join(str(x) for x in (\n                d / \"production_data/nse_official/bulk_eod\",\n                d / \"processed/universe_expansion_2026_v4/bulk_eod\",\n            ))\n        )\n\n    out = d / \"models/all_nse_delivery_multifactor_candidate_2026\"\n    log(f\"NSE bulk archive: {bulk}\")\n    log(f\"Candidate output: {out}\")\n    log(\"Step 1/2: building ALL-NSE 59-feature candidate...\")\n    env = dict(os.environ)\n    env[\"PYTHONUNBUFFERED\"] = \"1\"\n\n    build_cmd = [\n        sys.executable, \"-u\", \"colab_build_all_nse_delivery_multifactor_candidate.py\",\n        \"--bulk-root\", str(bulk), \"--out\", str(out),\n    ]\n    if a.refresh_fundamentals:\n        build_cmd.append(\"--refresh-fundamentals\")\n    if a.retry_failed_fundamentals:\n        build_cmd.append(\"--retry-failed-fundamentals\")\n    if a.force_rebuild_candidate:\n        build_cmd.append(\"--force-rebuild-candidate\")\n    subprocess.run(build_cmd, cwd=p, check=True, env=env)\n\n    log(\"Step 2/2: validating candidate artifacts...\")\n    validate_cmd = [\n        sys.executable, \"-u\", \"validate_multifactor_candidate.py\",\n        \"--model-dir\", str(out),\n    ]\n    subprocess.run(validate_cmd, cwd=p, check=True, env=env)\n\n    log(\"COLAB MULTIFACTOR CANDIDATE COMPLETE\")\n    log(f\"NSE bulk EOD: {bulk}\")\n    log(f\"Candidate: {out}\")\n    log(\"Production model: UNCHANGED\")\n    return 0\n\nif __name__ == \"__main__\":\n    main()\n","validate_multifactor_candidate.py":"#!/usr/bin/env python3\n\"\"\"Validate an ALL-NSE multi-factor candidate bundle.\"\"\"\nfrom __future__ import annotations\nimport argparse,hashlib,json\nfrom pathlib import Path\nimport joblib\nfrom multifactor_data_layer_v2 import FEATURES_59\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\"rb\") as f:\n        for b in iter(lambda:f.read(1024*1024),b\"\"): h.update(b)\n    return h.hexdigest()\n\ndef main():\n    if len(FEATURES_59) != 59:\n        raise SystemExit(f\"Repository feature contract is not 59 features: found {len(FEATURES_59)}\")\n    ap=argparse.ArgumentParser(); ap.add_argument(\"--model-dir\",default=\"/content/drive/MyDrive/Indian_Market_Research/models/all_nse_delivery_multifactor_candidate_2026\"); a=ap.parse_args()\n    root=Path(a.model_dir); mp=root/\"ALL_NSE_MULTIFACTOR_59_MODEL_MANIFEST_2026.json\"\n    if not mp.exists(): raise SystemExit(f\"Missing manifest: {mp}\")\n    m=json.loads(mp.read_text(encoding=\"utf-8\"))\n    if m.get(\"promotion_status\")!=\"CANDIDATE_ONLY\": raise SystemExit(\"Refusing non-candidate bundle\")\n    if m.get(\"features\")!=FEATURES_59: raise SystemExit(\"59-feature contract mismatch\")\n    expected_hash=hashlib.sha256(json.dumps(FEATURES_59,separators=(\",\",\":\")).encode()).hexdigest()\n    manifest_hash=m.get(\"feature_contract_hash\")\n    if manifest_hash is not None and manifest_hash!=expected_hash:\n        raise SystemExit(\"59-feature contract hash mismatch\")\n    report={\"engine_version\":m.get(\"engine_version\"),\"promotion_status\":m.get(\"promotion_status\"),\"universe\":m.get(\"universe\"),\"artifacts\":{}}\n    for name,item in m.get(\"artifacts\",{}).items():\n        report[\"artifacts\"][name]=item\n        if not isinstance(item,dict) or item.get(\"status\")!=\"CANDIDATE_READY\": continue\n        for kind in (\"classifier\",\"regressor\"):\n            p=Path(item[kind])\n            if not p.exists(): raise SystemExit(f\"Missing {kind}: {p}\")\n            model=joblib.load(p); steps=getattr(model,\"named_steps\",{}) ; imp=steps.get(\"imputer\") if hasattr(steps,\"get\") else None\n            learned=int(getattr(imp,\"n_features_in_\",0)) if imp is not None else 0\n            if not learned and imp is not None and getattr(imp,\"statistics_\",None) is not None:\n                learned=int(len(imp.statistics_))\n            if learned!=59:\n                raise SystemExit(f\"{p}: expected 59 original features, found {learned}\")\n            names=getattr(model,\"feature_names_in_\",None)\n            if names is not None and list(names)!=FEATURES_59:\n                raise SystemExit(f\"{p}: learned feature-name contract does not match FEATURES_59\")\n            report[\"artifacts\"].setdefault(name,{})[kind+\"_sha256\"]=sha256(p)\n    out=root/\"CANDIDATE_VALIDATION_REPORT_2026.json\"; out.write_text(json.dumps(report,indent=2,default=str),encoding=\"utf-8\")\n    print(json.dumps(report,indent=2,default=str))\n\nif __name__==\"__main__\": main()\n"}

for name, content in FILES.items():
    (PROJECT / name).write_text(content, encoding='utf-8')

print("SOURCE BOOTSTRAP COMPLETE")
print("Project:", PROJECT)
for name in FILES:
    print(f'{name}: {(PROJECT / name).stat().st_size:,} bytes')


In [ ]:
from pathlib import Path
import subprocess, sys

PROJECT = Path("/content/indian-market-trading")
required = [
    "persistent_store.py",
    "nse_data_layer.py",
    "delivery_training_universe.py",
    "multifactor_data_layer_v2.py",
    "colab_build_all_nse_delivery_multifactor_candidate.py",
    "colab_run_multifactor_upgrade.py",
    "validate_multifactor_candidate.py",
]
for name in required:
    print(name, ':', (PROJECT / name).exists())

print("\nGPU:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)

print("\nSYNTAX:")
subprocess.run([sys.executable, '-m', 'py_compile'] + [str(PROJECT/n) for n in required], check=True)
print("SYNTAX CHECK PASSED")


In [ ]:
%cd /content/indian-market-trading
!python colab_run_multifactor_upgrade.py --preflight


## Candidate build
After the preflight passes, run:

`%cd /content/indian-market-trading`
`!python colab_run_multifactor_upgrade.py`
